## **Google Maps Data Pre-Proccesing**

### **Data preparation**

In [3]:
import pandas as pd
from pathlib import Path

# ===================== PATHS =====================
input_root = Path(r"C:\Users\aws12\Desktop\Pre-Proccesing Step -GP 2\Google Maps Data")
output_root = Path(r"C:\Users\aws12\Desktop\Pre-Proccesing Step -GP 2\After Cleaning")
output_root.mkdir(parents=True, exist_ok=True)

# ===================== REGION MAPPING =====================
FOLDER_TO_REGION = {
    "منطقة الباحة": "Al-Baha",
    "منطقة عسير": "Aseer",
    "منطقة جازان": "Jazan",
    "منطقة نجران": "Najran",

}

# ===================== TARGET COLUMN ORDER =====================
desired_order = [
    "CategoryName",
    "Location/lat",
    "Location/lng",
    "Neighborhood",
    "Date",
    "Stars",
    "City",
    "Text",
    "Place Name",
    "Street",
    "Region"
]

# ===================== RENAME MAP =====================
rename_map = {
    "categoryName": "CategoryName",
    "location/lat": "Location/lat",
    "location/lng": "Location/lng",
    "neighborhood": "Neighborhood",
    "publishedAtDate": "Date",
    "stars": "Stars",
    "city": "City",
    "text": "Text",
    "textTranslated": "TextTranslated",
    "street": "Street",
    "title": "Place Name",
}

processed = 0
skipped = 0

# ===================== PIPELINE =====================
for folder_name, region_value in FOLDER_TO_REGION.items():
    region_folder = input_root / folder_name

    if not region_folder.exists():
        print(f"⚠️ Missing folder: {region_folder}")
        continue

    out_region_folder = output_root / folder_name
    out_region_folder.mkdir(parents=True, exist_ok=True)

    for file in region_folder.rglob("*.xlsx"):
        try:
            print(f"📄 Processing: {file.name}")

            df = pd.read_excel(file)

            # --- Normalize column names ---
            df.columns = [str(c).strip() for c in df.columns]

            # --- Rename columns ---
            df = df.rename(columns={c: rename_map[c] for c in df.columns if c in rename_map})

            # --- Unify Arabic Text ---
            if "TextTranslated" in df.columns and "Text" in df.columns:
                df["Text"] = df["TextTranslated"].fillna(df["Text"])
                df.drop(columns=["TextTranslated"], inplace=True)

            # --- Add Region column ---
            df["Region"] = region_value

            # --- Check required columns ---
            missing = [c for c in desired_order if c not in df.columns]
            if missing:
                print(f"⚠️ Skipped {file.name} | Missing: {missing}")
                skipped += 1
                continue

            # --- Reorder columns ---
            df_final = df[desired_order].copy()

            # --- Save output ---
            out_file = out_region_folder / f"{file.stem}_final.xlsx"
            df_final.to_excel(out_file, index=False)

            processed += 1
            print(f"✅ Saved: {out_file.name}")

        except Exception as e:
            print(f"❌ Error in {file.name}: {e}")
            skipped += 1

print(f"\n🎉 DONE | Processed: {processed} | Skipped: {skipped}")

📄 Processing: أكواخ سار الريفية.xlsx
✅ Saved: أكواخ سار الريفية_final.xlsx
📄 Processing: القرية الأثرية بالأطاولة.xlsx
✅ Saved: القرية الأثرية بالأطاولة_final.xlsx
📄 Processing: جبل شدا الأعلى.xlsx
✅ Saved: جبل شدا الأعلى_final.xlsx
📄 Processing: حديقة الأمير سلطان بن سلمان.xlsx
✅ Saved: حديقة الأمير سلطان بن سلمان_final.xlsx
📄 Processing: حديقة الأمير محمد بن سعود.xlsx
✅ Saved: حديقة الأمير محمد بن سعود_final.xlsx
📄 Processing: حديقة الجسر النابتية.xlsx
✅ Saved: حديقة الجسر النابتية_final.xlsx
📄 Processing: حديقة الحسام ( حديقة شهبة ).xlsx
✅ Saved: حديقة الحسام ( حديقة شهبة )_final.xlsx
📄 Processing: حديقة الخزامى.xlsx
✅ Saved: حديقة الخزامى_final.xlsx
📄 Processing: حديقة الشمال.xlsx
✅ Saved: حديقة الشمال_final.xlsx
📄 Processing: حديقة الغرير بالمندق.xlsx
✅ Saved: حديقة الغرير بالمندق_final.xlsx
📄 Processing: حديقة الفراشة بالمندق.xlsx
✅ Saved: حديقة الفراشة بالمندق_final.xlsx
📄 Processing: حديقة و ممشى دانة بلجرشي.xlsx
✅ Saved: حديقة و ممشى دانة بلجرشي_final.xlsx
📄 Processing: شاليها

### **Data pre-processing**


In [4]:
import pandas as pd
from pathlib import Path
import re
import unicodedata

# ===================== PATHS =====================
input_root = Path(r"C:\Users\aws12\Desktop\Pre-Proccesing Step -GP 2\Google Maps Data")
output_root = Path(r"C:\Users\aws12\Desktop\Pre-Proccesing Step -GP 2\After Cleaning v3")
output_root.mkdir(parents=True, exist_ok=True)

# ===================== SETTINGS =====================
TEXT_COL = "Text"
MIN_WORDS = 2

# Source columns in your files
RAW_COL = "text"
TR_COL = "textTranslated"  # Arabic translation

# Transformer settings (lighter cleaning)
REMOVE_LATIN_TR = True
KEEP_DIGITS_TR = True

# ML settings
REMOVE_LATIN_ML = True
KEEP_DIGITS_ML = True   # keep numbers/dates
REMOVE_STOPWORDS_ML = True
BIND_NEGATION_ML = True  # ✅ recommended for sentiment with TF-IDF

# Add emoji sentiment token to ML text
ADD_EMO_TOKEN_TO_ML = True  # ✅ recommended

# ===================== REGEX PATTERNS =====================
URL_RE = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)
EMAIL_RE = re.compile(r"\b[\w\.-]+@[\w\.-]+\.\w+\b")
MENTION_RE = re.compile(r"@\w+")
HASHTAG_RE = re.compile(r"#(\w+)")
TATWEEL_RE = re.compile(r"\u0640")
DIACRITICS_RE = re.compile(r"[\u0617-\u061A\u064B-\u0652\u0657-\u065F\u0670\u06D6-\u06ED]")
PUNCT_SYMBOLS_RE = re.compile(r"[^\w\s\u0600-\u06FF]")  # keep Arabic letters + digits + underscore + spaces
MULTISPACE_RE = re.compile(r"\s+")
LATIN_RE = re.compile(r"[A-Za-z]+")
DIGITS_RE = re.compile(r"\d+")

# Emoji matcher (single emoji tokens; important: NO '+')
EMOJI_RE = re.compile(
    "[" +
    "\U0001F300-\U0001FAFF" +
    "\U00002700-\U000027BF" +
    "\U00002600-\U000026FF" +
    "]",
    flags=re.UNICODE
)

# IMPORTANT: compress repeated ARABIC LETTERS only (does NOT touch digits like 5000)
AR_LETTER_REPEATS = re.compile(r"([\u0600-\u06FF])\1{2,}")

# ===================== EMOJI SENTIMENT (expand later) =====================
POS_EMOJI = set(list("😀😃😄😁😆😊😍🥰😘😇🙂😉🤩😎😺😸😹👍👏🙌💯🔥⭐️🌟✨❤️🩵💚💛💜🤍"))
NEG_EMOJI = set(list("😞😔😟😕🙁☹️😣😖😫😩😢😭😤😠😡🤬👎💔💩🤢🤮😒😓😥😰😨😱"))

# ===================== STOPWORDS (sentiment-safe) =====================
NEGATION_WORDS = set("ما لا لم لن ليس مو مش بدون".split())
INTENSIFIERS = set("جدا جدًا مره مرة كثير كتير للغاية للغايه جدًاا جدااa".split())

AR_STOPWORDS = set("""
في من على إلى عن هذا هذه ذلك تلك هناك هنا كان كانت يكون تكون
مع أو ثم حيث الذي التي الذين اللواتي اذا إذ قد كل بعض أيضا فقط حتى بعد قبل عند بين
انه انها هم هن نحن انت انتم انا
""".split())

AR_STOPWORDS = AR_STOPWORDS - NEGATION_WORDS - INTENSIFIERS

# ===================== HELPERS =====================
def normalize_unicode(text: str) -> str:
    return unicodedata.normalize("NFKC", str(text))

def remove_noise(text: str) -> str:
    text = URL_RE.sub(" ", text)
    text = EMAIL_RE.sub(" ", text)
    text = MENTION_RE.sub(" ", text)
    text = HASHTAG_RE.sub(r" \1 ", text)  # keep hashtag word without '#'
    return text

def arabic_normalize_light(text: str) -> str:
    text = DIACRITICS_RE.sub("", text)
    text = TATWEEL_RE.sub("", text)
    text = re.sub(r"[إأٱآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    return text

def remove_punct(text: str) -> str:
    return PUNCT_SYMBOLS_RE.sub(" ", text)

def remove_emojis_from_text(text: str) -> str:
    return EMOJI_RE.sub(" ", text)

def squeeze_repeats_safe(text: str) -> str:
    return AR_LETTER_REPEATS.sub(r"\1", text)

def finalize(text: str) -> str:
    return MULTISPACE_RE.sub(" ", str(text)).strip()

def extract_emojis(text: str):
    if pd.isna(text):
        return []
    return EMOJI_RE.findall(str(text))

def emoji_counts(emoji_list):
    if not emoji_list:
        return 0, 0
    pos = sum(e in POS_EMOJI for e in emoji_list)
    neg = sum(e in NEG_EMOJI for e in emoji_list)
    return pos, neg

def emoji_sentiment(emoji_list):
    if not emoji_list:
        return "NEU"
    pos = sum(e in POS_EMOJI for e in emoji_list)
    neg = sum(e in NEG_EMOJI for e in emoji_list)
    if pos > neg:
        return "POS"
    if neg > pos:
        return "NEG"
    return "NEU"

def emoji_score(pos_count: int, neg_count: int) -> int:
    return int(pos_count - neg_count)

def bind_negation(text: str) -> str:
    toks = text.split()
    out = []
    i = 0
    while i < len(toks):
        if toks[i] in NEGATION_WORDS and i + 1 < len(toks):
            out.append(toks[i] + "_" + toks[i + 1])
            i += 2
        else:
            out.append(toks[i])
            i += 1
    return " ".join(out)

def tr_cleanup(text: str) -> str:
    if REMOVE_LATIN_TR:
        text = LATIN_RE.sub(" ", text)
    if not KEEP_DIGITS_TR:
        text = DIGITS_RE.sub(" ", text)
    return finalize(text)

def ml_cleanup(text: str) -> str:
    if REMOVE_LATIN_ML:
        text = LATIN_RE.sub(" ", text)
    if not KEEP_DIGITS_ML:
        text = DIGITS_RE.sub(" ", text)

    text = finalize(text)

    if BIND_NEGATION_ML:
        text = bind_negation(text)

    if REMOVE_STOPWORDS_ML:
        toks = [t for t in text.split() if (t not in AR_STOPWORDS) and (len(t) > 1)]
        text = " ".join(toks)

    return finalize(text)

def wc(s: str) -> int:
    s = str(s).strip()
    return 0 if not s else len(s.split())

# ===================== BATCH PROCESS =====================
processed = 0
failed = 0
changed_numbers_total_files = 0

for file in input_root.rglob("*.xlsx"):
    try:
        df = pd.read_excel(file)

        # ===================== DROP NON-ESSENTIAL COLUMNS =====================
        cat_cols = [c for c in df.columns if str(c).startswith("categories/")]
        drop_cols = cat_cols + [c for c in ["countryCode", "name"] if c in df.columns]
        df.drop(columns=drop_cols, inplace=True, errors="ignore")

        # ===================== FORCE ARABIC TRANSLATION INTO MAIN TEXT =====================
        # Ensure RAW_COL exists
        if RAW_COL not in df.columns:
            df[RAW_COL] = pd.NA

        # Keep original raw text
        df["Text_Orig"] = df[RAW_COL]

        # Build unified Arabic-only Text
        if TR_COL in df.columns:
            df[TEXT_COL] = df[TR_COL].fillna(df[RAW_COL]).astype(str)
            df.drop(columns=[TR_COL], inplace=True, errors="ignore")
        else:
            df[TEXT_COL] = df[RAW_COL].astype(str)

        raw = df[TEXT_COL].fillna("").astype(str)

        # ---- Emoji features ----
        df["Emoji_List"] = raw.apply(extract_emojis)
        df["Emoji_Count"] = df["Emoji_List"].apply(len)

        df["Emoji_Pos_Count"], df["Emoji_Neg_Count"] = zip(*df["Emoji_List"].apply(emoji_counts))
        df["Emoji_Score"] = df.apply(lambda r: emoji_score(r["Emoji_Pos_Count"], r["Emoji_Neg_Count"]), axis=1)

        df["Emoji_Sentiment"] = df["Emoji_List"].apply(emoji_sentiment)

        # ---- Base clean ----
        df["Text_Base"] = (
            raw.apply(normalize_unicode)
               .apply(remove_noise)
               .apply(remove_emojis_from_text)   # emojis stored in Emoji_List
               .apply(arabic_normalize_light)
               .apply(remove_punct)
               .apply(squeeze_repeats_safe)
               .apply(finalize)
        )

        # ---- Numeric integrity check (internal only) ----
        nums_raw = raw.apply(lambda s: DIGITS_RE.findall(s))
        nums_base = df["Text_Base"].apply(lambda s: DIGITS_RE.findall(s))
        changed_numbers_rows = (nums_raw != nums_base).sum()
        if changed_numbers_rows > 0:
            changed_numbers_total_files += 1
            print(f"⚠️ Numbers changed in file: {file.name} | rows affected: {changed_numbers_rows}")

        # ---- Two outputs ----
        df["Text_TR"] = df["Text_Base"].apply(tr_cleanup)
        df["Text_ML"] = df["Text_Base"].apply(ml_cleanup)

        # ---- Add emoji sentiment token to ML text ----
        if ADD_EMO_TOKEN_TO_ML:
            emo_token = df["Emoji_Sentiment"].map({"POS": "EMO_POS", "NEG": "EMO_NEG", "NEU": "EMO_NEU"}).fillna("EMO_NEU")
            df["Text_ML"] = (df["Text_ML"] + " " + emo_token).apply(finalize)

        # ---- Short flags ----
        df["Is_Short_TR"] = df["Text_TR"].apply(wc) < MIN_WORDS
        df["Is_Short_ML"] = df["Text_ML"].apply(wc) < MIN_WORDS

        # ---- Save output preserving folder structure ----
        rel = file.relative_to(input_root)
        out_file = output_root / rel.parent / f"{file.stem}_textready{file.suffix}"
        out_file.parent.mkdir(parents=True, exist_ok=True)
        df.to_excel(out_file, index=False)

        processed += 1
        print(f"✅ Saved: {out_file}")

    except Exception as e:
        failed += 1
        print(f"❌ Failed: {file.name} | {e}")

print(f"\n🎉 DONE | Processed: {processed} | Failed: {failed} | Files with number-changes warnings: {changed_numbers_total_files}")


⚠️ Numbers changed in file: أكواخ سار الريفية.xlsx | rows affected: 2
✅ Saved: C:\Users\aws12\Desktop\Pre-Proccesing Step -GP 2\After Cleaning v3\منطقة الباحة\أكواخ سار الريفية_textready.xlsx
✅ Saved: C:\Users\aws12\Desktop\Pre-Proccesing Step -GP 2\After Cleaning v3\منطقة الباحة\القرية الأثرية بالأطاولة_textready.xlsx
✅ Saved: C:\Users\aws12\Desktop\Pre-Proccesing Step -GP 2\After Cleaning v3\منطقة الباحة\جبل شدا الأعلى_textready.xlsx
⚠️ Numbers changed in file: حديقة الأمير سلطان بن سلمان.xlsx | rows affected: 1
✅ Saved: C:\Users\aws12\Desktop\Pre-Proccesing Step -GP 2\After Cleaning v3\منطقة الباحة\حديقة الأمير سلطان بن سلمان_textready.xlsx
⚠️ Numbers changed in file: حديقة الأمير محمد بن سعود.xlsx | rows affected: 1
✅ Saved: C:\Users\aws12\Desktop\Pre-Proccesing Step -GP 2\After Cleaning v3\منطقة الباحة\حديقة الأمير محمد بن سعود_textready.xlsx
✅ Saved: C:\Users\aws12\Desktop\Pre-Proccesing Step -GP 2\After Cleaning v3\منطقة الباحة\حديقة الجسر النابتية_textready.xlsx
⚠️ Numbers chan

## **Checking The Proccesing Work** 

### **Imports**

In [5]:
import pandas as pd
import numpy as np
import re

raw_path   = r"C:\Users\aws12\Desktop\Pre-Proccesing Step -GP 2\Google Maps Data\منطقة الباحة\منتزة غابة رغدان.xlsx"
ready_path = r"C:\Users\aws12\Desktop\Pre-Proccesing Step -GP 2\After Cleaning v3\منطقة الباحة\منتزة غابة رغدان_textready.xlsx"

raw_df = pd.read_excel(raw_path)
ready_df = pd.read_excel(ready_path)

print("RAW:", raw_df.shape)
print("READY:", ready_df.shape)


RAW: (37938, 16)
READY: (37938, 23)


### **Show sample data**


In [6]:
display(raw_df.head(3))
display(ready_df.head(3))

print("\nRAW columns:\n", list(raw_df.columns))
print("\nREADY columns:\n", list(ready_df.columns))


,categories/0,categories/1,categories/2,categoryName,city,countryCode,location/lat,location/lng,name,neighborhood,stars,publishedAtDate,text,textTranslated,title,street
0,متنزه,غابة قومية,مزار سياحي,متنزه,الباحة,SA,20.020606,41.432788,Waheed Khalid,NaN,5,2025-10-27T14:56:49.177Z,تم تطويره عن السابق و الشيء المميز لا يوجد قرو...,NaN,منتزه غابة رغدان,4657 الحكم الزروقي، 6905
1,متنزه,غابة قومية,مزار سياحي,متنزه,الباحة,SA,20.020606,41.432788,سلطان المالكي,NaN,5,2025-10-27T09:01:04.964Z,NaN,NaN,منتزه غابة رغدان,4657 الحكم الزروقي، 6905
2,متنزه,غابة قومية,مزار سياحي,متنزه,الباحة,SA,20.020606,41.432788,mrdi qarni,NaN,5,2025-10-26T01:13:00.119Z,الاول,NaN,منتزه غابة رغدان,4657 الحكم الزروقي، 6905


,categoryName,city,location/lat,location/lng,neighborhood,stars,publishedAtDate,text,title,street,...,Emoji_Count,Emoji_Pos_Count,Emoji_Neg_Count,Emoji_Score,Emoji_Sentiment,Text_Base,Text_TR,Text_ML,Is_Short_TR,Is_Short_ML
0,متنزه,الباحة,20.020606,41.432788,NaN,5,2025-10-27T14:56:49.177Z,تم تطويره عن السابق و الشيء المميز لا يوجد قرو...,منتزه غابة رغدان,4657 الحكم الزروقي، 6905,...,0,0,0,0,NEU,تم تطويره عن السابق و الشيء المميز لا يوجد قرو...,تم تطويره عن السابق و الشيء المميز لا يوجد قرو...,تم تطويره السابق الشيء المميز لا_يوجد قرود الم...,False,False
1,متنزه,الباحة,20.020606,41.432788,NaN,5,2025-10-27T09:01:04.964Z,NaN,منتزه غابة رغدان,4657 الحكم الزروقي، 6905,...,0,0,0,0,NEU,NaN,NaN,EMO_NEU,True,True
2,متنزه,الباحة,20.020606,41.432788,NaN,5,2025-10-26T01:13:00.119Z,الاول,منتزه غابة رغدان,4657 الحكم الزروقي، 6905,...,0,0,0,0,NEU,الاول,الاول,الاول EMO_NEU,True,False



RAW columns:
 ['categories/0', 'categories/1', 'categories/2', 'categoryName', 'city', 'countryCode', 'location/lat', 'location/lng', 'name', 'neighborhood', 'stars', 'publishedAtDate', 'text', 'textTranslated', 'title', 'street']

READY columns:
 ['categoryName', 'city', 'location/lat', 'location/lng', 'neighborhood', 'stars', 'publishedAtDate', 'text', 'title', 'street', 'Text_Orig', 'Text', 'Emoji_List', 'Emoji_Count', 'Emoji_Pos_Count', 'Emoji_Neg_Count', 'Emoji_Score', 'Emoji_Sentiment', 'Text_Base', 'Text_TR', 'Text_ML', 'Is_Short_TR', 'Is_Short_ML']


### **Table of deleted and added columns**


In [7]:
raw_cols = set(raw_df.columns.astype(str))
ready_cols = set(ready_df.columns.astype(str))

dropped = sorted(list(raw_cols - ready_cols))
added   = sorted(list(ready_cols - raw_cols))

col_changes = pd.DataFrame({
    "Dropped Columns (اختفت)": pd.Series(dropped),
    "Added Columns (انضافت)": pd.Series(added)
})

col_changes


,Dropped Columns (اختفت),Added Columns (انضافت)
0,categories/0,Emoji_Count
1,categories/1,Emoji_List
2,categories/2,Emoji_Neg_Count
3,countryCode,Emoji_Pos_Count
4,name,Emoji_Score
5,textTranslated,Emoji_Sentiment
6,NaN,Is_Short_ML
7,NaN,Is_Short_TR
8,NaN,Text
9,NaN,Text_Base


### **Summary of the number of records and columns (before/after)**


In [8]:
overview = pd.DataFrame({
    "Dataset": ["Before (RAW)", "After (READY)"],
    "Rows": [len(raw_df), len(ready_df)],
    "Columns": [len(raw_df.columns), len(ready_df.columns)]
})
overview


,Dataset,Rows,Columns
0,Before (RAW),37938,16
1,After (READY),37938,23


### **Percentage of empty text before/after**


In [9]:
def empty_ratio(series):
    s = series.fillna("").astype(str).str.strip()
    return round(((s == "") | (s.str.lower().isin(["nan","none"]))).mean()*100, 2)

# أعمدة النص حسب ملفاتكم
raw_text_col = "text" if "text" in raw_df.columns else None
raw_tr_col   = "textTranslated" if "textTranslated" in raw_df.columns else None

ready_text_col = "Text" if "Text" in ready_df.columns else None
base_col = "Text_Base" if "Text_Base" in ready_df.columns else None
ml_col   = "Text_ML" if "Text_ML" in ready_df.columns else None
tr_col   = "Text_TR" if "Text_TR" in ready_df.columns else None

summary_empty = pd.DataFrame({
    "Column": [raw_text_col, raw_tr_col, ready_text_col, base_col, ml_col, tr_col],
    "Empty_%": [
        empty_ratio(raw_df[raw_text_col]) if raw_text_col else np.nan,
        empty_ratio(raw_df[raw_tr_col]) if raw_tr_col else np.nan,
        empty_ratio(ready_df[ready_text_col]) if ready_text_col else np.nan,
        empty_ratio(ready_df[base_col]) if base_col else np.nan,
        empty_ratio(ready_df[ml_col]) if ml_col else np.nan,
        empty_ratio(ready_df[tr_col]) if tr_col else np.nan,
    ]
})
summary_empty


,Column,Empty_%
0,text,44.16
1,textTranslated,96.18
2,Text,44.16
3,Text_Base,44.26
4,Text_ML,0.00
5,Text_TR,44.26


### **Text length before/after (number of words/letters)**


In [10]:
def length_stats(series):
    s = series.fillna("").astype(str).str.strip()
    words = s.apply(lambda x: len(x.split()))
    chars = s.str.len()
    return pd.Series({
        "avg_words": round(words.mean(), 2),
        "median_words": float(words.median()),
        "max_words": int(words.max()),
        "avg_chars": round(chars.mean(), 2),
        "median_chars": float(chars.median()),
        "max_chars": int(chars.max())
    })

stats = {}

if raw_text_col:
    stats["Before_raw(text)"] = length_stats(raw_df[raw_text_col])
if raw_tr_col:
    stats["Before_translated(textTranslated)"] = length_stats(raw_df[raw_tr_col])
if ready_text_col:
    stats["After_Text"] = length_stats(ready_df[ready_text_col])
if base_col:
    stats["After_Text_Base"] = length_stats(ready_df[base_col])
if ml_col:
    stats["After_Text_ML"] = length_stats(ready_df[ml_col])
if tr_col:
    stats["After_Text_TR"] = length_stats(ready_df[tr_col])

pd.DataFrame(stats).T


,avg_words,median_words,max_words,avg_chars,median_chars,max_chars
Before_raw(text),5.49,1.0,363.0,31.08,7.0,2243.0
Before_translated(textTranslated),0.36,0.0,277.0,2.08,0.0,1749.0
After_Text,5.43,1.0,363.0,30.76,7.0,2243.0
After_Text_Base,5.38,1.0,366.0,30.13,6.0,2219.0
After_Text_ML,5.83,2.0,321.0,36.09,14.0,2086.0
After_Text_TR,5.36,1.0,366.0,30.06,6.0,2219.0


### **Percentage of Latin letters before/after (verification of Arabic standardization)**

In [11]:
latin_re = re.compile(r"[A-Za-z]")

def latin_present_ratio(series):
    s = series.fillna("").astype(str)
    has_latin = s.apply(lambda x: bool(latin_re.search(x)))
    return round(has_latin.mean()*100, 2)

latin_report = pd.DataFrame({
    "Column": [raw_text_col, raw_tr_col, ready_text_col, base_col, ml_col, tr_col],
    "Latin_present_%": [
        latin_present_ratio(raw_df[raw_text_col]) if raw_text_col else np.nan,
        latin_present_ratio(raw_df[raw_tr_col]) if raw_tr_col else np.nan,
        latin_present_ratio(ready_df[ready_text_col]) if ready_text_col else np.nan,
        latin_present_ratio(ready_df[base_col]) if base_col else np.nan,
        latin_present_ratio(ready_df[ml_col]) if ml_col else np.nan,
        latin_present_ratio(ready_df[tr_col]) if tr_col else np.nan,
    ]
})
latin_report


,Column,Latin_present_%
0,text,3.60
1,textTranslated,0.06
2,Text,0.20
3,Text_Base,0.19
4,Text_ML,100.00
5,Text_TR,0.00


### **Emoji summary (how many comments it contains + distribution of emotions)**


In [12]:
emoji_summary = {}

if "Emoji_Count" in ready_df.columns:
    emoji_summary["Reviews_with_emoji_%"] = round((ready_df["Emoji_Count"] > 0).mean()*100, 2)

if "Emoji_Sentiment" in ready_df.columns:
    emoji_summary["Emoji_Sentiment_Distribution"] = ready_df["Emoji_Sentiment"].value_counts(dropna=False)

emoji_summary


{'Reviews_with_emoji_%': 0.79,
 'Emoji_Sentiment_Distribution': Emoji_Sentiment
 NEU    37765
 POS      171
 NEG        2
 Name: count, dtype: int64}

### **Before/After Example Table**


In [15]:
def text_diff_score(a, b):
    a = "" if pd.isna(a) else str(a)
    b = "" if pd.isna(b) else str(b)
    return abs(len(a) - len(b))

ready_df["Diff_Raw_vs_Base"] = ready_df["Text_Orig"].fillna("").str.len() - ready_df["Text_Base"].fillna("").str.len()
ready_df["Diff_Raw_vs_ML"]   = ready_df["Text_Orig"].fillna("").str.len() - ready_df["Text_ML"].fillna("").str.len()

ready_df[["Diff_Raw_vs_Base", "Diff_Raw_vs_ML"]].describe()


,Diff_Raw_vs_Base,Diff_Raw_vs_ML
count,37938.000000,37938.000000
mean,0.972350,-4.990115
std,6.482819,10.117192
min,-37.000000,-27.000000
25%,0.000000,-8.000000
50%,0.000000,-7.000000
75%,0.000000,-5.000000
max,418.000000,552.000000


In [16]:
examples_df = (
    ready_df[
        (ready_df["Text_Orig"].notna()) &
        (ready_df["Text_Orig"].str.len() > 20) &
        (
            (ready_df["Diff_Raw_vs_Base"].abs() > 15) |
            (ready_df["Diff_Raw_vs_ML"].abs() > 15) |
            (ready_df["Emoji_Count"] > 0)
        )
    ]
    .sample(10, random_state=42)
)

examples_df.shape


(10, 25)

In [18]:
comparison_table = pd.DataFrame({
    "Before (Raw Text)": examples_df["Text_Orig"],
    "After (Text_Base)": examples_df["Text_Base"],
    "ML Processing (Text_ML)": examples_df["Text_ML"],
    "Pretrained Processing (Text_TR)": examples_df["Text_TR"],
    "Emoji_Sentiment": examples_df["Emoji_Sentiment"]
})

comparison_table


,Before (Raw Text),After (Text_Base),ML Processing (Text_ML),Pretrained Processing (Text_TR),Emoji_Sentiment
30192,اجمل حديقه بالمملكه❤️,اجمل حديقه بالمملكه,اجمل حديقه بالمملكه EMO_POS,اجمل حديقه بالمملكه,POS
2714,من افضل الاماكن بس ياريت في الصيفيه يعملوا موا...,من افضل الاماكن بس ياريت في الصيفيه يعملوا موا...,افضل الاماكن بس ياريت الصيفيه يعملوا مواقف خار...,من افضل الاماكن بس ياريت في الصيفيه يعملوا موا...,NEU
20183,منتزه غابة رغدان حقا مكان يستحق الزيارة للترفي...,منتزه غابة رغدان حقا مكان يستحق الزيارة للترفي...,منتزه غابة رغدان حقا مكان يستحق الزيارة للترفي...,منتزه غابة رغدان حقا مكان يستحق الزيارة للترفي...,NEU
16998,"Amazing experience, a must visit place if you ...",تجربة رائعة، وجهة لا تفوت عند زيارة الباحة منا...,تجربة رائعة، وجهة لا_تفوت زيارة الباحة مناظر ط...,تجربة رائعة، وجهة لا تفوت عند زيارة الباحة منا...,NEU
22259,"Amazing place, there’s many ice cream trucks, ...",مكان رائع، فيه عربات ايس كريم كثيرة، ومطعم واح...,مكان رائع، فيه عربات ايس كريم كثيرة، ومطعم واح...,مكان رائع، فيه عربات ايس كريم كثيرة، ومطعم واح...,NEU
3782,Nice plane with good activities especially zip...,طائرة رائعة مع انشطة ممتعة، لا سيما الانزلاق ب...,طائرة رائعة انشطة ممتعة، لا_سيما الانزلاق بالح...,طائرة رائعة مع انشطة ممتعة، لا سيما الانزلاق ب...,NEU
15852,ك منتزه يعتبر من افضل المنتزهات ف منطقة الباحة...,ك منتزه يعتبر من افضل المنتزهات ف منطقة الباحة...,منتزه يعتبر افضل المنتزهات منطقة الباحة لكن تع...,ك منتزه يعتبر من افضل المنتزهات ف منطقة الباحة...,NEU
3401,غابة جميلة ورايقة وهادئة وتصلح للعوائل وفيها أ...,غابة جميلة ورايقة وهادئة وتصلح للعوائل وفيها ا...,غابة جميلة ورايقة وهادئة وتصلح للعوائل وفيها ا...,غابة جميلة ورايقة وهادئة وتصلح للعوائل وفيها ا...,POS
17540,Very beautiful place to visit in Al Bahah. We ...,مكان جميل جدا للزيارة في الباحة استمتعنا بتجرب...,مكان جميل جدا للزيارة الباحة استمتعنا بتجربة ا...,مكان جميل جدا للزيارة في الباحة استمتعنا بتجرب...,NEU
12512,مكان جميل جداا اعجز عن التعبير عنه صراحة❤️,مكان جميل جداا اعجز عن التعبير عنه صراحة,مكان جميل جداا اعجز التعبير عنه صراحة EMO_POS,مكان جميل جداا اعجز عن التعبير عنه صراحة,POS


## **Tik Tok Data Pre-Proccesing**

In [1]:
import pandas as pd
from pathlib import Path
import re
import unicodedata

# ===================== PATHS =====================
input_root = Path(r"C:\Users\aws12\Desktop\Pre-Proccesing Step -GP 2\Tik Tok Datasets - before Cleaning")
output_root = Path(r"C:\Users\aws12\Desktop\Pre-Proccesing Step -GP 2\Tik Tok Data - After Cleaning")
output_root.mkdir(parents=True, exist_ok=True)

# ===================== SETTINGS =====================
RAW_TEXT_COL = "text"
TEXT_COL = "Text"
MIN_WORDS = 2

# If translation exists, we will use it (Arabic only)
TRANSLATION_CANDIDATES = ["Text_translated", "textTranslated"]

# Arabic-only policy
ARABIC_ONLY = True

# Transformer settings
REMOVE_LATIN_TR = True   # ✅ Arabic only
KEEP_DIGITS_TR = True

# ML settings
REMOVE_LATIN_ML = True   # ✅ Arabic only
KEEP_DIGITS_ML = True
REMOVE_STOPWORDS_ML = True
BIND_NEGATION_ML = True
ADD_EMO_TOKEN_TO_ML = True

# Drop columns (privacy / not useful)
DROP_COLS_EXACT = ["uid", "avatarThumbnail", "uniqueId"]

# ===================== REGEX PATTERNS =====================
URL_RE = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)
EMAIL_RE = re.compile(r"\b[\w\.-]+@[\w\.-]+\.\w+\b")
MENTION_RE = re.compile(r"@\w+")
HASHTAG_RE = re.compile(r"#(\w+)")
TATWEEL_RE = re.compile(r"\u0640")
DIACRITICS_RE = re.compile(r"[\u0617-\u061A\u064B-\u0652\u0657-\u065F\u0670\u06D6-\u06ED]")
PUNCT_SYMBOLS_RE = re.compile(r"[^\w\s\u0600-\u06FF]")  # keep Arabic letters + digits + underscore + spaces
MULTISPACE_RE = re.compile(r"\s+")
LATIN_RE = re.compile(r"[A-Za-z]+")
DIGITS_RE = re.compile(r"\d+")

EMOJI_RE = re.compile(
    "[" +
    "\U0001F300-\U0001FAFF" +
    "\U00002700-\U000027BF" +
    "\U00002600-\u26FF" +
    "]",
    flags=re.UNICODE
)

AR_LETTER_REPEATS = re.compile(r"([\u0600-\u06FF])\1{2,}")

POS_EMOJI = set(list("😀😃😄😁😆😊😍🥰😘😇🙂😉🤩😎😺😸😹👍👏🙌💯🔥⭐️🌟✨❤️🩵💚💛💜🤍"))
NEG_EMOJI = set(list("😞😔😟😕🙁☹️😣😖😫😩😢😭😤😠😡🤬👎💔💩🤢🤮😒😓😥😰😨😱"))

NEGATION_WORDS = set("ما لا لم لن ليس مو مش بدون".split())
INTENSIFIERS = set("جدا جدًا مره مرة كثير كتير للغاية للغايه جدًاا جدااa".split())

AR_STOPWORDS = set("""
في من على إلى عن هذا هذه ذلك تلك هناك هنا كان كانت يكون تكون
مع أو ثم حيث الذي التي الذين اللواتي اذا إذ قد كل بعض أيضا فقط حتى بعد قبل عند بين
انه انها هم هن نحن انت انتم انا
""".split())

AR_STOPWORDS = AR_STOPWORDS - NEGATION_WORDS - INTENSIFIERS

# ===================== HELPERS =====================
def normalize_unicode(text: str) -> str:
    return unicodedata.normalize("NFKC", str(text))

def remove_noise(text: str) -> str:
    text = URL_RE.sub(" ", text)
    text = EMAIL_RE.sub(" ", text)
    text = MENTION_RE.sub(" ", text)
    text = HASHTAG_RE.sub(r" \1 ", text)
    return text

def arabic_normalize_light(text: str) -> str:
    text = DIACRITICS_RE.sub("", text)
    text = TATWEEL_RE.sub("", text)
    text = re.sub(r"[إأٱآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    return text

def remove_punct(text: str) -> str:
    return PUNCT_SYMBOLS_RE.sub(" ", text)

def remove_emojis_from_text(text: str) -> str:
    return EMOJI_RE.sub(" ", text)

def squeeze_repeats_safe(text: str) -> str:
    return AR_LETTER_REPEATS.sub(r"\1", text)

def finalize(text: str) -> str:
    return MULTISPACE_RE.sub(" ", str(text)).strip()

def extract_emojis(text: str):
    if pd.isna(text):
        return []
    return EMOJI_RE.findall(str(text))

def emoji_counts(emoji_list):
    if not emoji_list:
        return 0, 0
    pos = sum(e in POS_EMOJI for e in emoji_list)
    neg = sum(e in NEG_EMOJI for e in emoji_list)
    return pos, neg

def emoji_sentiment(emoji_list):
    if not emoji_list:
        return "NEU"
    pos = sum(e in POS_EMOJI for e in emoji_list)
    neg = sum(e in NEG_EMOJI for e in emoji_list)
    if pos > neg:
        return "POS"
    if neg > pos:
        return "NEG"
    return "NEU"

def emoji_score(pos_count: int, neg_count: int) -> int:
    return int(pos_count - neg_count)

def bind_negation(text: str) -> str:
    toks = text.split()
    out = []
    i = 0
    while i < len(toks):
        if toks[i] in NEGATION_WORDS and i + 1 < len(toks):
            out.append(toks[i] + "_" + toks[i + 1])
            i += 2
        else:
            out.append(toks[i])
            i += 1
    return " ".join(out)

def tr_cleanup(text: str) -> str:
    if REMOVE_LATIN_TR:
        text = LATIN_RE.sub(" ", text)
    if not KEEP_DIGITS_TR:
        text = DIGITS_RE.sub(" ", text)
    return finalize(text)

def ml_cleanup(text: str) -> str:
    if REMOVE_LATIN_ML:
        text = LATIN_RE.sub(" ", text)
    if not KEEP_DIGITS_ML:
        text = DIGITS_RE.sub(" ", text)

    text = finalize(text)

    if BIND_NEGATION_ML:
        text = bind_negation(text)

    if REMOVE_STOPWORDS_ML:
        toks = [t for t in text.split() if (t not in AR_STOPWORDS) and (len(t) > 1)]
        text = " ".join(toks)

    return finalize(text)

def wc(s: str) -> int:
    s = str(s).strip()
    return 0 if not s else len(s.split())

# ===================== BATCH PROCESS =====================
processed = 0
failed = 0

for file in input_root.rglob("*.xlsx"):
    try:
        df = pd.read_excel(file)

        # Drop privacy/unneeded columns
        df.drop(columns=[c for c in DROP_COLS_EXACT if c in df.columns], inplace=True, errors="ignore")

        # Ensure RAW_TEXT_COL exists
        if RAW_TEXT_COL not in df.columns:
            df[RAW_TEXT_COL] = pd.NA

        # Keep original
        df["Text_Orig"] = df[RAW_TEXT_COL]

        # Use translation column if available
        tr_col = next((c for c in TRANSLATION_CANDIDATES if c in df.columns), None)

        if tr_col:
            df[TEXT_COL] = df[tr_col].fillna(df[RAW_TEXT_COL]).astype(str)
            df.drop(columns=[tr_col], inplace=True, errors="ignore")
        else:
            df[TEXT_COL] = df[RAW_TEXT_COL].fillna("").astype(str)

        raw = df[TEXT_COL].fillna("").astype(str)

        # Emoji features
        df["Emoji_List"] = raw.apply(extract_emojis)
        df["Emoji_Count"] = df["Emoji_List"].apply(len)
        df["Emoji_Pos_Count"], df["Emoji_Neg_Count"] = zip(*df["Emoji_List"].apply(emoji_counts))
        df["Emoji_Score"] = df.apply(lambda r: emoji_score(r["Emoji_Pos_Count"], r["Emoji_Neg_Count"]), axis=1)
        df["Emoji_Sentiment"] = df["Emoji_List"].apply(emoji_sentiment)

        # Base clean
        df["Text_Base"] = (
            raw.apply(normalize_unicode)
               .apply(remove_noise)
               .apply(remove_emojis_from_text)
               .apply(arabic_normalize_light)
               .apply(remove_punct)
               .apply(squeeze_repeats_safe)
               .apply(finalize)
        )

        # Two outputs
        df["Text_TR"] = df["Text_Base"].apply(tr_cleanup)
        df["Text_ML"] = df["Text_Base"].apply(ml_cleanup)

        # Add emoji sentiment token to ML
        if ADD_EMO_TOKEN_TO_ML:
            emo_token = df["Emoji_Sentiment"].map({"POS": "EMO_POS", "NEG": "EMO_NEG", "NEU": "EMO_NEU"}).fillna("EMO_NEU")
            df["Text_ML"] = (df["Text_ML"] + " " + emo_token).apply(finalize)

        # Short flags
        df["Is_Short_TR"] = df["Text_TR"].apply(wc) < MIN_WORDS
        df["Is_Short_ML"] = df["Text_ML"].apply(wc) < MIN_WORDS

        # Save output
        rel = file.relative_to(input_root)
        out_file = output_root / rel.parent / f"{file.stem}_textready{file.suffix}"
        out_file.parent.mkdir(parents=True, exist_ok=True)
        df.to_excel(out_file, index=False)

        processed += 1
        print(f"✅ Saved: {out_file}")

    except Exception as e:
        failed += 1
        print(f"❌ Failed: {file.name} | {e}")

print(f"\n🎉 DONE | Processed: {processed} | Failed: {failed}")


✅ Saved: C:\Users\aws12\Desktop\Pre-Proccesing Step -GP 2\Tik Tok Data - After Cleaning\الأماكن السياحية Tik Tok - منطقة الباحة\Video comments 1_textready.xlsx
✅ Saved: C:\Users\aws12\Desktop\Pre-Proccesing Step -GP 2\Tik Tok Data - After Cleaning\الأماكن السياحية Tik Tok - منطقة الباحة\Video comments 10_textready.xlsx
✅ Saved: C:\Users\aws12\Desktop\Pre-Proccesing Step -GP 2\Tik Tok Data - After Cleaning\الأماكن السياحية Tik Tok - منطقة الباحة\Video comments 11_textready.xlsx
✅ Saved: C:\Users\aws12\Desktop\Pre-Proccesing Step -GP 2\Tik Tok Data - After Cleaning\الأماكن السياحية Tik Tok - منطقة الباحة\Video comments 12_textready.xlsx
✅ Saved: C:\Users\aws12\Desktop\Pre-Proccesing Step -GP 2\Tik Tok Data - After Cleaning\الأماكن السياحية Tik Tok - منطقة الباحة\Video comments 13_textready.xlsx
✅ Saved: C:\Users\aws12\Desktop\Pre-Proccesing Step -GP 2\Tik Tok Data - After Cleaning\الأماكن السياحية Tik Tok - منطقة الباحة\Video comments 14_textready.xlsx
✅ Saved: C:\Users\aws12\Desktop\Pre

### **Read all Excel files from the Results folder**


In [2]:
import pandas as pd
from pathlib import Path

# ضع مسار مجلد البيانات بعد المعالجة هنا
folder_path = Path(r"C:\Users\aws12\Desktop\Pre-Proccesing Step -GP 2\Tik Tok Data - After Cleaning")

all_files = list(folder_path.rglob("*.xlsx"))

print("عدد الملفات:", len(all_files))
all_files[:5]  # عرض أول 5 ملفات


عدد الملفات: 103


[WindowsPath('C:/Users/aws12/Desktop/Pre-Proccesing Step -GP 2/Tik Tok Data - After Cleaning/الأماكن السياحية Tik Tok - منطقة الباحة/Video comments 10_textready.xlsx'),
 WindowsPath('C:/Users/aws12/Desktop/Pre-Proccesing Step -GP 2/Tik Tok Data - After Cleaning/الأماكن السياحية Tik Tok - منطقة الباحة/Video comments 11_textready.xlsx'),
 WindowsPath('C:/Users/aws12/Desktop/Pre-Proccesing Step -GP 2/Tik Tok Data - After Cleaning/الأماكن السياحية Tik Tok - منطقة الباحة/Video comments 12_textready.xlsx'),
 WindowsPath('C:/Users/aws12/Desktop/Pre-Proccesing Step -GP 2/Tik Tok Data - After Cleaning/الأماكن السياحية Tik Tok - منطقة الباحة/Video comments 13_textready.xlsx'),
 WindowsPath('C:/Users/aws12/Desktop/Pre-Proccesing Step -GP 2/Tik Tok Data - After Cleaning/الأماكن السياحية Tik Tok - منطقة الباحة/Video comments 14_textready.xlsx')]

### **Merge all files into a single DataFrame**


In [3]:
dfs = []

for file in all_files:
    try:
        df = pd.read_excel(file)
        df["source_file"] = file.name   # حفظ اسم الملف كمصدر
        dfs.append(df)
    except Exception as e:
        print("Error reading:", file.name, "|", e)

merged_df = pd.concat(dfs, ignore_index=True)

print("Merged Shape:", merged_df.shape)
merged_df.head(3)


Error reading: ~$Video comments 1_textready.xlsx | [Errno 13] Permission denied: 'C:\\Users\\aws12\\Desktop\\Pre-Proccesing Step -GP 2\\Tik Tok Data - After Cleaning\\الأماكن السياحية Tik Tok - منطقة الباحة\\~$Video comments 1_textready.xlsx'
Merged Shape: (43051, 20)


,text,diggCount,replyCommentTotal,createTimeISO,videoWebUrl,cid,Text_Orig,Text,Emoji_List,Emoji_Count,Emoji_Pos_Count,Emoji_Neg_Count,Emoji_Score,Emoji_Sentiment,Text_Base,Text_TR,Text_ML,Is_Short_TR,Is_Short_ML,source_file
0,كلها خلصتها بيوم,5,5.0,2025-07-14T22:08:46.000Z,https://www.tiktok.com/@uelc4/video/7519396488...,7527062899476153344,كلها خلصتها بيوم,كلها خلصتها بيوم,[],0,0,0,0,NEU,كلها خلصتها بيوم,كلها خلصتها بيوم,كلها خلصتها بيوم EMO_NEU,False,False,Video comments 10_textready.xlsx
1,من الفضاوه,1,NaN,2025-07-19T20:43:55.000Z,https://www.tiktok.com/@uelc4/video/7519396488...,7528896524617565184,من الفضاوه,من الفضاوه,[],0,0,0,0,NEU,من الفضاوه,من الفضاوه,الفضاوه EMO_NEU,False,False,Video comments 10_textready.xlsx
2,تحمست للسفره محتارين ابها ولا الباحه,5,16.0,2025-06-24T06:37:36.000Z,https://www.tiktok.com/@uelc4/video/7519396488...,7519401291077911552,تحمست للسفره محتارين ابها ولا الباحه,تحمست للسفره محتارين ابها ولا الباحه,[],0,0,0,0,NEU,تحمست للسفره محتارين ابها ولا الباحه,تحمست للسفره محتارين ابها ولا الباحه,تحمست للسفره محتارين ابها ولا الباحه EMO_NEU,False,False,Video comments 10_textready.xlsx


### **Number of comments within each file**


In [4]:
file_summary = merged_df.groupby("source_file").size().reset_index(name="comments_count")
file_summary.sort_values("comments_count", ascending=False)


,source_file,comments_count
6,Video comments 16_textready.xlsx,2950
0,Video comments 10_textready.xlsx,2400
29,Video comments 4_textready.xlsx,2305
10,Video comments 1_textready.xlsx,1915
14,Video comments 23_textready.xlsx,1914
28,Video comments 3_textready.xlsx,1910
32,Video comments 7_textready.xlsx,1813
8,Video comments 18_textready.xlsx,1791
34,Video comments 9_textready.xlsx,1776
2,Video comments 12_textready.xlsx,1699


### **Emoji ratio + distribution**

In [8]:
import pandas as pd
import numpy as np
import re
from collections import Counter

# اختر واحد فقط:
# data = df         # لو ملف واحد
data = merged_df     # لو دمجت كل ملفات المجلد


In [9]:
emoji_ratio = round((data["Emoji_Count"] > 0).mean() * 100, 2)
emoji_count = int((data["Emoji_Count"] > 0).sum())
emoji_dist  = data["Emoji_Sentiment"].value_counts(dropna=False)

print("Emoji ratio %:", emoji_ratio)
print("Comments with emoji:", emoji_count)
emoji_dist


Emoji ratio %: 5.99
Comments with emoji: 2577


Emoji_Sentiment
NEU    41455
POS     1545
NEG       51
Name: count, dtype: int64

### **Top Emojis**

In [10]:
all_emojis = data["Emoji_List"].dropna().sum()
top_emojis = pd.DataFrame(Counter(all_emojis).most_common(15), columns=["Emoji", "Count"])
top_emojis


,Emoji,Count
0,[,43051
1,],43051
2,',9878
3,❤,2443
4,",",2362
5,,2362
6,♥,857
7,✨,438
8,♀,102
9,✅,100


### **Top words + bigrams From Text_ML**


#### **Top Words**

In [11]:
ml_clean = (
    data["Text_ML"].fillna("").astype(str)
    .str.replace(r"\bEMO_(POS|NEG|NEU)\b", "", regex=True)
    .str.replace(r"\s+", " ", regex=True).str.strip()
)

words = " ".join(ml_clean).split()
top_words = pd.DataFrame(Counter(words).most_common(30), columns=["Word", "Count"])
top_words


,Word,Count
0,الله,3694
1,بس,1890
2,ابها,1844
3,علي,1558
4,ولا,1374
5,والله,1314
6,فيها,1277
7,الي,1267
8,فيه,1218
9,شي,1166


#### **Top Bigrams**

In [12]:
tokens = " ".join(ml_clean).split()
bigrams = list(zip(tokens, tokens[1:]))
top_bigrams = pd.DataFrame(
    [(" ".join(k), v) for k, v in Counter(bigrams).most_common(20)],
    columns=["Bigram", "Count"]
)
top_bigrams


,Bigram,Count
0,ماشاء الله,439
1,تبارك الله,291
2,الله تبارك,230
3,السلام عليكم,207
4,خميس مشيط,195
5,شاء الله,192
6,ان شاء,172
7,ما_شاء الله,172
8,سبحان الله,158
9,باذن الله,138


### **Duplicate ratio (Spam detection) Using Text_Base**

In [13]:
base = data["Text_Base"].fillna("").astype(str).str.strip()
dup_count = base.duplicated().sum()
dup_ratio = round((dup_count / len(data)) * 100, 2)

print("Duplicate comments:", int(dup_count))
print("Duplicate ratio %:", dup_ratio)

top_repeated = base.value_counts().head(15).reset_index()
top_repeated.columns = ["Repeated_Comment(Text_Base)", "Frequency"]
top_repeated


Duplicate comments: 9180
Duplicate ratio %: 21.32


,Repeated_Comment(Text_Base),Frequency
0,,5052
1,ه,159
2,لا,106
3,ماشاءالله,73
4,الله,56
5,بكم,54
6,روعه,48
7,ماشاء الله,48
8,شكرا,45
9,يا اخي، انا مسلم ملتزم، اصلي خمس مرات يوميا جئ...,43


### **Hashtags + mentions (from the raw text)**

#### **Top Hashtags**

In [14]:
hashtags = data["text"].fillna("").astype(str).str.findall(r"#(\w+)")
hashtags_flat = hashtags.explode().dropna()

top_hashtags = hashtags_flat.value_counts().head(20).reset_index()
top_hashtags.columns = ["Hashtag", "Count"]
top_hashtags


,Hashtag,Count
0,اكسبلور,5
1,سكليف_فوري,3
2,شقق,3
3,ابها,3
4,خميس_مشيط,3
5,أبها,3
6,خميس,3
7,اكسبلورر,2
8,قحطان,2
9,نجران,2


#### **Top Mentions**

In [15]:
mentions = data["text"].fillna("").astype(str).str.findall(r"@(\w+)")
mentions_flat = mentions.explode().dropna()

top_mentions = mentions_flat.value_counts().head(20).reset_index()
top_mentions.columns = ["Mention", "Count"]
top_mentions


,Mention,Count
0,مزرعة,25
1,منتجع,25
2,R,21
3,N,17
4,SAMA,16
5,M,16
6,محمد,14
7,ابو,13
8,S,12
9,A,11


### **Engagement stats (likes/replies)**

In [16]:
engagement_stats = pd.DataFrame({
    "Metric": [
        "Total Comments",
        "Avg Likes", "Median Likes", "Max Likes",
        "Avg Replies", "Median Replies", "Max Replies"
    ],
    "Value": [
        len(data),
        round(data["diggCount"].mean(), 2),
        float(data["diggCount"].median()),
        int(data["diggCount"].max()),
        round(data["replyCommentTotal"].mean(), 2),
        float(data["replyCommentTotal"].median()),
        int(data["replyCommentTotal"].max()),
    ]
})
engagement_stats


,Metric,Value
0,Total Comments,43051.00
1,Avg Likes,1.67
2,Median Likes,0.00
3,Max Likes,1396.00
4,Avg Replies,0.50
5,Median Replies,0.00
6,Max Replies,141.00


### **Top 20 Viral Comments**

In [17]:
top_viral = data.sort_values("diggCount", ascending=False)[
    ["Text_Orig", "Text_ML", "diggCount", "replyCommentTotal", "Emoji_Sentiment"]
].head(20)

top_viral


,Text_Orig,Text_ML,diggCount,replyCommentTotal,Emoji_Sentiment
1497,ياخي هالمنطقة زرتها السنة اللي فاتت والى الآن ...,ياخي هالمنطقة زرتها السنة اللي فاتت والي الان ...,1396,100.0,NEU
10361,والله الباحة لو اطلع بحوشنا اعتبر نفسي تمشيت,والله الباحة لو اطلع بحوشنا اعتبر نفسي تمشيت E...,698,11.0,NEU
6029,وفالاخير اروح غابة رغدان,وفالاخير اروح غابة رغدان EMO_NEU,628,14.0,NEU
36266,ههههههههههههههههههههههههه مضحك تعاملهم معاها و...,مضحك تعاملهم معاها وقديش مبسوطين EMO_NEU,465,65.0,NEU
10313,+ قرية ذي عين + منتزة الامير حسام + مهرجان الا...,قرية ذي عين منتزة الامير حسام مهرجان الاطاولة ...,425,24.0,NEU
21657,الطايف موسمهم ماحطوه بفلوس !!!,الطايف موسمهم ماحطوه بفلوس EMO_NEU,412,7.0,NEU
25357,ترا فخ عشان تطيحون و تخف زحمة ابها,ترا فخ عشان تطيحون تخف زحمة ابها EMO_NEU,408,26.0,NEU
34129,اشكرك على ذكر المسميات الصحيحه لكل شى فعلا انت...,اشكرك علي ذكر المسميات الصحيحه لكل شي فعلا نمو...,383,4.0,NEU
11635,الباحه تجنننن❤️تحتاج فنادق وشقق زياده,الباحه تجن تحتاج فنادق وشقق زياده EMO_POS,335,0.0,POS
6030,اسم المكان اكواخ الحازم ..طلعوه اكسبلور,اسم المكان اكواخ الحازم طلعوه اكسبلور EMO_NEU,333,9.0,NEU


### **Time-based distribution (From createTimeISO)**

In [18]:
data["createTimeISO"] = pd.to_datetime(data["createTimeISO"], errors="coerce")

# يومي
data["date"] = data["createTimeISO"].dt.date
daily = data.groupby("date").size().reset_index(name="comments_count").sort_values("comments_count", ascending=False)
daily.head(20)


,date,comments_count
1000,2025-06-28,658
922,2025-04-11,483
1056,2025-08-23,477
999,2025-06-27,405
1058,2025-08-25,394
1001,2025-06-29,392
583,2024-05-07,361
1002,2025-06-30,357
997,2025-06-25,353
1057,2025-08-24,352


In [19]:
# شهري
data["month"] = data["createTimeISO"].dt.to_period("M").astype(str)
monthly = data.groupby("month").size().reset_index(name="comments_count").sort_values("month")
monthly


C:\Users\aws12\AppData\Local\Temp\ipykernel_10576\1345400247.py:2: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  data["month"] = data["createTimeISO"].dt.to_period("M").astype(str)


,month,comments_count
0,2021-06,405
1,2021-07,101
2,2021-08,293
3,2021-09,11
4,2021-10,4
5,2021-11,1
6,2021-12,5
7,2022-01,8
8,2022-02,7
9,2022-03,5


## **YouTube Pre-Proccesing Datasets**

In [20]:
import pandas as pd
from pathlib import Path
import re
import unicodedata

# ===================== PATHS =====================
input_root = Path(r"C:\Users\aws12\Desktop\Pre-Proccesing Step -GP 2\Raw Datasets - YouTube")      # 📌 ضع ملفات اليوتيوب الخام هنا
output_root = Path(r"C:\Users\aws12\Desktop\Pre-Proccesing Step -GP 2\YouTube - Cleaned Data")  # 📌 هنا تُحفظ الملفات بعد المعالجة
output_root.mkdir(parents=True, exist_ok=True)

# ===================== SETTINGS =====================
RAW_TEXT_COL = "comment"   # في ملفك: comment
TEXT_COL = "Text"
MIN_WORDS = 2

# Arabic-only policy (مثل Google Maps/TikTok)
KEEP_DIGITS_TR = True
KEEP_DIGITS_ML = True

REMOVE_STOPWORDS_ML = True
BIND_NEGATION_ML = True
ADD_EMO_TOKEN_TO_ML = True

# Columns to drop (privacy / not useful)
DROP_COLS_EXACT = ["author", "type", "commentsCount"]

# Optional: remove YouTube spam words (subscribe/like/channel...)
REMOVE_YT_SPAM = True

# ===================== REGEX PATTERNS =====================
URL_RE = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)
EMAIL_RE = re.compile(r"\b[\w\.-]+@[\w\.-]+\.\w+\b")
MENTION_RE = re.compile(r"@\w+")
HASHTAG_RE = re.compile(r"#(\w+)")
TATWEEL_RE = re.compile(r"\u0640")
DIACRITICS_RE = re.compile(r"[\u0617-\u061A\u064B-\u0652\u0657-\u065F\u0670\u06D6-\u06ED]")
PUNCT_SYMBOLS_RE = re.compile(r"[^\w\s\u0600-\u06FF]")  # keep Arabic letters + digits + underscore + spaces
MULTISPACE_RE = re.compile(r"\s+")
DIGITS_RE = re.compile(r"\d+")

# Remove any non-Arabic letters (Arabic-only guarantee)
NON_ARABIC_LETTERS_RE = re.compile(r"[^\u0600-\u06FF\s\d_]")

# YouTube timestamps like 1:23 or 12:34 or 1:02:33
TIMESTAMP_RE = re.compile(r"\b\d{1,2}:\d{2}(?::\d{2})?\b")

# Common YouTube spam-ish keywords (Arabic + English)
YT_SPAM_RE = re.compile(
    r"\b("
    r"subscribe|sub|like|channel|follow|share|comment|"
    r"اشترك|اشتراك|لايك|اعجاب|قناتي|قناة|تابع|متابعة|"
    r"فعاليات\s*الجرس|جرس|شير"
    r")\b",
    re.IGNORECASE
)

# Emoji matcher (single emoji tokens)
EMOJI_RE = re.compile(
    "[" +
    "\U0001F300-\U0001FAFF" +
    "\U00002700-\U000027BF" +
    "\U00002600-\u26FF" +
    "]",
    flags=re.UNICODE
)

# compress repeated Arabic letters only
AR_LETTER_REPEATS = re.compile(r"([\u0600-\u06FF])\1{2,}")

# ===================== SIMPLE EMOJI SENTIMENT =====================
POS_EMOJI = set(list("😀😃😄😁😆😊😍🥰😘😇🙂😉🤩😎😺😸😹👍👏🙌💯🔥⭐️🌟✨❤️🩵💚💛💜🤍"))
NEG_EMOJI = set(list("😞😔😟😕🙁☹️😣😖😫😩😢😭😤😠😡🤬👎💔💩🤢🤮😒😓😥😰😨😱"))

# ===================== AR STOPWORDS (sentiment-safe) =====================
NEGATION_WORDS = set("ما لا لم لن ليس مو مش بدون".split())
INTENSIFIERS = set("جدا جدًا مره مرة كثير كتير للغاية للغايه".split())

AR_STOPWORDS = set("""
في من على إلى عن هذا هذه ذلك تلك هناك هنا كان كانت يكون تكون
مع أو ثم حيث الذي التي الذين اللواتي اذا إذ قد كل بعض أيضا فقط حتى بعد قبل عند بين
انه انها هم هن نحن انت انتم انا
""".split())

AR_STOPWORDS = AR_STOPWORDS - NEGATION_WORDS - INTENSIFIERS

# ===================== HELPERS =====================
def normalize_unicode(text: str) -> str:
    return unicodedata.normalize("NFKC", str(text))

def remove_noise(text: str) -> str:
    text = URL_RE.sub(" ", text)
    text = EMAIL_RE.sub(" ", text)
    text = MENTION_RE.sub(" ", text)
    text = HASHTAG_RE.sub(r" \1 ", text)
    text = TIMESTAMP_RE.sub(" ", text)
    if REMOVE_YT_SPAM:
        text = YT_SPAM_RE.sub(" ", text)
    return text

def arabic_normalize_light(text: str) -> str:
    text = DIACRITICS_RE.sub("", text)
    text = TATWEEL_RE.sub("", text)
    text = re.sub(r"[إأٱآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    return text

def remove_punct(text: str) -> str:
    return PUNCT_SYMBOLS_RE.sub(" ", text)

def remove_emojis_from_text(text: str) -> str:
    return EMOJI_RE.sub(" ", text)

def squeeze_repeats_safe(text: str) -> str:
    return AR_LETTER_REPEATS.sub(r"\1", text)

def finalize(text: str) -> str:
    return MULTISPACE_RE.sub(" ", str(text)).strip()

def extract_emojis(text: str):
    if pd.isna(text):
        return []
    return EMOJI_RE.findall(str(text))

def emoji_counts(emoji_list):
    if not emoji_list:
        return 0, 0
    pos = sum(e in POS_EMOJI for e in emoji_list)
    neg = sum(e in NEG_EMOJI for e in emoji_list)
    return pos, neg

def emoji_sentiment(emoji_list):
    if not emoji_list:
        return "NEU"
    pos = sum(e in POS_EMOJI for e in emoji_list)
    neg = sum(e in NEG_EMOJI for e in emoji_list)
    if pos > neg:
        return "POS"
    if neg > pos:
        return "NEG"
    return "NEU"

def emoji_score(pos_count: int, neg_count: int) -> int:
    return int(pos_count - neg_count)

def bind_negation(text: str) -> str:
    toks = text.split()
    out = []
    i = 0
    while i < len(toks):
        if toks[i] in NEGATION_WORDS and i + 1 < len(toks):
            out.append(toks[i] + "_" + toks[i + 1])
            i += 2
        else:
            out.append(toks[i])
            i += 1
    return " ".join(out)

# ===================== CLEANUP FUNCTIONS =====================
def tr_cleanup(text: str) -> str:
    # Arabic-only guarantee
    text = NON_ARABIC_LETTERS_RE.sub(" ", str(text))

    if not KEEP_DIGITS_TR:
        text = DIGITS_RE.sub(" ", text)

    return finalize(text)

def ml_cleanup(text: str) -> str:
    # Arabic-only guarantee
    text = NON_ARABIC_LETTERS_RE.sub(" ", str(text))

    if not KEEP_DIGITS_ML:
        text = DIGITS_RE.sub(" ", text)

    text = finalize(text)

    if BIND_NEGATION_ML:
        text = bind_negation(text)

    if REMOVE_STOPWORDS_ML:
        toks = [t for t in text.split() if (t not in AR_STOPWORDS) and (len(t) > 1)]
        text = " ".join(toks)

    return finalize(text)

def wc(s: str) -> int:
    s = str(s).strip()
    return 0 if not s else len(s.split())

# ===================== BATCH PROCESS =====================
processed = 0
failed = 0

for file in input_root.rglob("*.xlsx"):
    try:
        df = pd.read_excel(file)

        # Drop privacy/unneeded columns
        df.drop(columns=[c for c in DROP_COLS_EXACT if c in df.columns], inplace=True, errors="ignore")

        # Ensure text column exists
        if RAW_TEXT_COL not in df.columns:
            df[RAW_TEXT_COL] = pd.NA

        # Keep original + unify
        df["Text_Orig"] = df[RAW_TEXT_COL]
        df[TEXT_COL] = df[RAW_TEXT_COL].fillna("").astype(str)

        raw = df[TEXT_COL].fillna("").astype(str)

        # Emoji features
        df["Emoji_List"] = raw.apply(extract_emojis)
        df["Emoji_Count"] = df["Emoji_List"].apply(len)
        df["Emoji_Pos_Count"], df["Emoji_Neg_Count"] = zip(*df["Emoji_List"].apply(emoji_counts))
        df["Emoji_Score"] = df.apply(lambda r: emoji_score(r["Emoji_Pos_Count"], r["Emoji_Neg_Count"]), axis=1)
        df["Emoji_Sentiment"] = df["Emoji_List"].apply(emoji_sentiment)

        # Base clean
        df["Text_Base"] = (
            raw.apply(normalize_unicode)
               .apply(remove_noise)
               .apply(remove_emojis_from_text)
               .apply(arabic_normalize_light)
               .apply(remove_punct)
               .apply(squeeze_repeats_safe)
               .apply(finalize)
        )

        # Two outputs
        df["Text_TR"] = df["Text_Base"].apply(tr_cleanup)
        df["Text_ML"] = df["Text_Base"].apply(ml_cleanup)

        # Add emoji sentiment token to ML
        if ADD_EMO_TOKEN_TO_ML:
            emo_token = df["Emoji_Sentiment"].map({"POS": "EMO_POS", "NEG": "EMO_NEG", "NEU": "EMO_NEU"}).fillna("EMO_NEU")
            df["Text_ML"] = (df["Text_ML"] + " " + emo_token).apply(finalize)

        # Short flags
        df["Is_Short_TR"] = df["Text_TR"].apply(wc) < MIN_WORDS
        df["Is_Short_ML"] = df["Text_ML"].apply(wc) < MIN_WORDS

        # Fill NaN in text columns
        for c in ["Text_Base", "Text_TR", "Text_ML"]:
            df[c] = df[c].fillna("")

        # Save output preserving folder structure
        rel = file.relative_to(input_root)
        out_file = output_root / rel.parent / f"{file.stem}_textready{file.suffix}"
        out_file.parent.mkdir(parents=True, exist_ok=True)
        df.to_excel(out_file, index=False)

        processed += 1
        print(f"✅ Saved: {out_file}")

    except Exception as e:
        failed += 1
        print(f"❌ Failed: {file.name} | {e}")

print(f"\n🎉 DONE | Processed: {processed} | Failed: {failed}")


✅ Saved: C:\Users\aws12\Desktop\Pre-Proccesing Step -GP 2\YouTube - Cleaned Data\YouTube Datasets - Al-Baha\Al-Baha Vedio Comments 1_textready.xlsx
✅ Saved: C:\Users\aws12\Desktop\Pre-Proccesing Step -GP 2\YouTube - Cleaned Data\YouTube Datasets - Al-Baha\Al-Baha Vedio Comments 10_textready.xlsx
✅ Saved: C:\Users\aws12\Desktop\Pre-Proccesing Step -GP 2\YouTube - Cleaned Data\YouTube Datasets - Al-Baha\Al-Baha Vedio Comments 11_textready.xlsx
✅ Saved: C:\Users\aws12\Desktop\Pre-Proccesing Step -GP 2\YouTube - Cleaned Data\YouTube Datasets - Al-Baha\Al-Baha Vedio Comments 12_textready.xlsx
✅ Saved: C:\Users\aws12\Desktop\Pre-Proccesing Step -GP 2\YouTube - Cleaned Data\YouTube Datasets - Al-Baha\Al-Baha Vedio Comments 13_textready.xlsx
✅ Saved: C:\Users\aws12\Desktop\Pre-Proccesing Step -GP 2\YouTube - Cleaned Data\YouTube Datasets - Al-Baha\Al-Baha Vedio Comments 14_textready.xlsx
✅ Saved: C:\Users\aws12\Desktop\Pre-Proccesing Step -GP 2\YouTube - Cleaned Data\YouTube Datasets - Al-Baha

### **Statistics and Results**

### **Locate the path and retrieve all files**


In [21]:
import pandas as pd
from pathlib import Path

# ضع مسار مجلد اليوتيوب بعد المعالجة هنا
folder_path = Path(r"C:\Users\aws12\Desktop\Pre-Proccesing Step -GP 2\YouTube - Cleaned Data")

all_files = list(folder_path.rglob("*_textready.xlsx"))

print("عدد ملفات YouTube المعالجة:", len(all_files))
all_files[:5]   # عرض أول 5 ملفات


عدد ملفات YouTube المعالجة: 82


[WindowsPath('C:/Users/aws12/Desktop/Pre-Proccesing Step -GP 2/YouTube - Cleaned Data/YouTube Datasets - Al-Baha/Al-Baha Vedio Comments 10_textready.xlsx'),
 WindowsPath('C:/Users/aws12/Desktop/Pre-Proccesing Step -GP 2/YouTube - Cleaned Data/YouTube Datasets - Al-Baha/Al-Baha Vedio Comments 11_textready.xlsx'),
 WindowsPath('C:/Users/aws12/Desktop/Pre-Proccesing Step -GP 2/YouTube - Cleaned Data/YouTube Datasets - Al-Baha/Al-Baha Vedio Comments 12_textready.xlsx'),
 WindowsPath('C:/Users/aws12/Desktop/Pre-Proccesing Step -GP 2/YouTube - Cleaned Data/YouTube Datasets - Al-Baha/Al-Baha Vedio Comments 13_textready.xlsx'),
 WindowsPath('C:/Users/aws12/Desktop/Pre-Proccesing Step -GP 2/YouTube - Cleaned Data/YouTube Datasets - Al-Baha/Al-Baha Vedio Comments 14_textready.xlsx')]

### **Read and merge all files into a single DataFrame**


In [22]:
dfs = []

for file in all_files:
    try:
        df = pd.read_excel(file)
        df["source_file"] = file.name      # حفظ اسم الملف كمصدر
        df["source_path"] = str(file)      # حفظ المسار (اختياري)
        dfs.append(df)
    except Exception as e:
        print("❌ Error reading:", file.name, "|", e)

merged_yt = pd.concat(dfs, ignore_index=True)

print("Merged YouTube Shape:", merged_yt.shape)
merged_yt.head(3)


Merged YouTube Shape: (22679, 21)


,comment,pageUrl,replyCount,title,videoID,voteCount,Text_Orig,Text,Emoji_List,Emoji_Count,...,Emoji_Neg_Count,Emoji_Score,Emoji_Sentiment,Text_Base,Text_TR,Text_ML,Is_Short_TR,Is_Short_ML,source_file,source_path
0,الحلق أو التقصير والحلق أفضل,https://youtu.be/L6q5SNP3zHg?si=9YKw_UE4vnNXx21W,0,حلق مع جبل شدا الأسفل - Fly with Shada Al-Asfa...,NaN,0,الحلق أو التقصير والحلق أفضل,الحلق أو التقصير والحلق أفضل,[],0,...,0,0,NEU,الحلق او التقصير والحلق افضل,الحلق او التقصير والحلق افضل,الحلق او التقصير والحلق افضل EMO_NEU,False,False,Al-Baha Vedio Comments 10_textready.xlsx,C:\Users\aws12\Desktop\Pre-Proccesing Step -GP...
1,الموسيقى❌❌❌❌ازعاج,https://youtu.be/L6q5SNP3zHg?si=9YKw_UE4vnNXx21W,0,حلق مع جبل شدا الأسفل - Fly with Shada Al-Asfa...,NaN,0,الموسيقى❌❌❌❌ازعاج,الموسيقى❌❌❌❌ازعاج,"['❌', '❌', '❌', '❌']",4,...,0,0,NEU,الموسيقي ازعاج,الموسيقي ازعاج,الموسيقي ازعاج EMO_NEU,False,False,Al-Baha Vedio Comments 10_textready.xlsx,C:\Users\aws12\Desktop\Pre-Proccesing Step -GP...
2,ليتك تحذف الموسيقى,https://youtu.be/L6q5SNP3zHg?si=9YKw_UE4vnNXx21W,0,حلق مع جبل شدا الأسفل - Fly with Shada Al-Asfa...,NaN,0,ليتك تحذف الموسيقى,ليتك تحذف الموسيقى,[],0,...,0,0,NEU,ليتك تحذف الموسيقي,ليتك تحذف الموسيقي,ليتك تحذف الموسيقي EMO_NEU,False,False,Al-Baha Vedio Comments 10_textready.xlsx,C:\Users\aws12\Desktop\Pre-Proccesing Step -GP...


### **Summary of the number of comments per file**


In [23]:
file_summary = merged_yt.groupby("source_file").size().reset_index(name="comments_count")
file_summary.sort_values("comments_count", ascending=False)


,source_file,comments_count
50,Jazan Vedio Comments 11_textready.xlsx,2098
5,Al-Baha Vedio Comments 15_textready.xlsx,2083
26,Aseer Vedio Comments 12_textready.xlsx,1144
67,Jazan Vedio Comments 3_textready.xlsx,1073
72,Jazan Vedio Comments 8_textready.xlsx,848
...,...,...
7,Al-Baha Vedio Comments 17_textready.xlsx,53
36,Aseer Vedio Comments 21_textready.xlsx,52
80,Najran Vedio Comments 7_textready.xlsx,52
43,Aseer Vedio Comments 4_textready.xlsx,46


### **Simple preparation and cleaning (for analysis)**


In [25]:
import pandas as pd
import numpy as np
import re
from collections import Counter

data = merged_yt.copy()

def empty_ratio(series):
    s = series.fillna("").astype(str).str.strip()
    return round(((s == "") | (s.str.lower().isin(["nan","none"]))).mean() * 100, 2)

def avg_words(series):
    return round(series.fillna("").astype(str).apply(lambda x: len(x.split())).mean(), 2)

# نسخة نظيفة من Text_ML بدون EMO tokens
ml_clean = (
    data["Text_ML"].fillna("").astype(str)
    .str.replace(r"\bEMO_(POS|NEG|NEU)\b", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)


### **Overview Of Data**

In [26]:
overview = pd.DataFrame({
    "Metric": ["Total Comments", "Total Files", "Total Columns"],
    "Value": [len(data), data["source_file"].nunique(), len(data.columns)]
})

file_summary = data.groupby("source_file").size().reset_index(name="comments_count") \
                   .sort_values("comments_count", ascending=False)

overview, file_summary.head(10)


(           Metric  Value
 0  Total Comments  22679
 1     Total Files     82
 2   Total Columns     21,
                                  source_file  comments_count
 50    Jazan Vedio Comments 11_textready.xlsx            2098
 5   Al-Baha Vedio Comments 15_textready.xlsx            2083
 26    Aseer Vedio Comments 12_textready.xlsx            1144
 67     Jazan Vedio Comments 3_textready.xlsx            1073
 72     Jazan Vedio Comments 8_textready.xlsx             848
 28    Aseer Vedio Comments 14_textready.xlsx             752
 23   Al-Baha Vedio Comments 9_textready.xlsx             620
 30    Aseer Vedio Comments 16_textready.xlsx             562
 31    Aseer Vedio Comments 17_textready.xlsx             531
 58    Jazan Vedio Comments 19_textready.xlsx             512)

### **Empty Text Stats**

In [27]:
empty_stats = pd.DataFrame({
    "Column": ["comment(raw)", "Text_Base", "Text_TR", "Text_ML"],
    "Empty_%": [
        empty_ratio(data.get("comment", pd.Series(dtype=str))),
        empty_ratio(data["Text_Base"]),
        empty_ratio(data["Text_TR"]),
        empty_ratio(data["Text_ML"])
    ]
})

empty_stats


,Column,Empty_%
0,comment(raw),1.67
1,Text_Base,3.71
2,Text_TR,6.32
3,Text_ML,0.00


### **Word Length Stats**

In [28]:
words_stats = pd.DataFrame({
    "Text_Version": ["Raw(comment)", "Text_Base", "Text_TR", "Text_ML"],
    "Avg_Words": [
        avg_words(data.get("comment", pd.Series(dtype=str))),
        avg_words(data["Text_Base"]),
        avg_words(data["Text_TR"]),
        avg_words(data["Text_ML"])
    ]
})

words_stats


,Text_Version,Avg_Words
0,Raw(comment),13.36
1,Text_Base,13.06
2,Text_TR,12.66
3,Text_ML,12.13


### **Emoji Stats + Top Emojis**

In [29]:
emoji_ratio = round((data["Emoji_Count"] > 0).mean() * 100, 2)
emoji_count = int((data["Emoji_Count"] > 0).sum())
emoji_dist = data["Emoji_Sentiment"].value_counts(dropna=False)

emoji_overview = pd.DataFrame({
    "Metric": ["Comments with Emoji %", "Comments with Emoji (count)"],
    "Value": [emoji_ratio, emoji_count]
})

all_emojis = data["Emoji_List"].dropna().sum()
top_emojis = pd.DataFrame(Counter(all_emojis).most_common(15), columns=["Emoji", "Count"])

emoji_overview, emoji_dist, top_emojis


(                        Metric    Value
 0        Comments with Emoji %    11.21
 1  Comments with Emoji (count)  2542.00,
 Emoji_Sentiment
 NEU    20617
 POS     2051
 NEG       11
 Name: count, dtype: int64,
    Emoji  Count
 0      [  22679
 1      ]  22679
 2      '  11572
 3      ❤   4837
 4      ,   3244
 5          3244
 6      ♥    320
 7      ✨     99
 8      ✋     57
 9      ✌     45
 10     ⚘     42
 11     ♂     37
 12     ♀     34
 13     ☝     31
 14     ❣     30)

### **Top Words + Top Bigrams**

In [30]:
# Top Words
words = " ".join(ml_clean).split()
top_words = pd.DataFrame(Counter(words).most_common(30), columns=["Word", "Count"])

# Top Bigrams
tokens = " ".join(ml_clean).split()
bigrams = list(zip(tokens, tokens[1:]))
top_bigrams = pd.DataFrame(
    [(" ".join(k), v) for k, v in Counter(bigrams).most_common(20)],
    columns=["Bigram", "Count"]
)

top_words, top_bigrams


(        Word  Count
 0       الله   6832
 1        علي   3036
 2      اليمن   1744
 3         ان   1646
 4        الي   1447
 5         يا   1362
 6      والله   1340
 7   السعودية   1240
 8         بس   1021
 9        ولا    998
 10     جازان    923
 11      فيها    892
 12        او    868
 13  السعوديه    862
 14      ابها    837
 15     ماشاء    679
 16     تبارك    655
 17      اللي    650
 18       الا    642
 19       اهل    635
 20        شي    631
 21      شكرا    595
 22       جدا    589
 23    الجنوب    583
 24       ابو    571
 25        لك    561
 26        لو    542
 27     جيزان    523
 28       حتي    507
 29      عليه    499,
               Bigram  Count
 0         ماشاء الله    657
 1         تبارك الله    419
 2         الله تبارك    407
 3        ما_شاء الله    388
 4         سبحان الله    299
 5           شاء الله    274
 6             ان شاء    235
 7          الله عليه    224
 8           العم علي    213
 9       تبارك الرحمن    198
 10         عليه وسلم    194


### **Duplicate Ratio**

In [31]:
base = data["Text_Base"].fillna("").astype(str).str.strip()
dup_count = base.duplicated().sum()
dup_ratio = round((dup_count / len(data)) * 100, 2)

dup_overview = pd.DataFrame({
    "Metric": ["Duplicate Comments", "Duplicate Ratio %"],
    "Value": [int(dup_count), dup_ratio]
})

top_repeated = base.value_counts().head(15).reset_index()
top_repeated.columns = ["Repeated_Comment(Text_Base)", "Frequency"]

dup_overview, top_repeated


(               Metric    Value
 0  Duplicate Comments  2422.00
 1   Duplicate Ratio %    10.68,
                           Repeated_Comment(Text_Base)  Frequency
 0                                                            842
 1                                                   ه         72
 2                                            وين يوسف         62
 3                                          ماشاء الله         33
 4                                                 اول         30
 5                                           ماشاءالله         30
 6                                         ما شاء الله         27
 7                               ماشاء الله تبارك الله         24
 8                                             شكرا لك         18
 9   لا ما عمرها كانت لليمن جنوب السعوديه سعودي من ...         18
 10                                         سبحان الله         18
 11                                              استمر         17
 12                                         ا

### **Engagement Stats (Likes/Replies) + Top Comments**

In [32]:
# إذا عندك ملفات ليس فيها الأعمدة، نتعامل بأمان:
vote = data["voteCount"] if "voteCount" in data.columns else pd.Series([np.nan]*len(data))
reply = data["replyCount"] if "replyCount" in data.columns else pd.Series([np.nan]*len(data))

engagement_stats = pd.DataFrame({
    "Metric": [
        "Avg Likes", "Median Likes", "Max Likes",
        "Avg Replies", "Median Replies", "Max Replies"
    ],
    "Value": [
        round(vote.mean(), 2),
        float(vote.median()) if vote.notna().any() else np.nan,
        int(vote.max()) if vote.notna().any() else np.nan,
        round(reply.mean(), 2),
        float(reply.median()) if reply.notna().any() else np.nan,
        int(reply.max()) if reply.notna().any() else np.nan
    ]
})

top_likes = data.sort_values("voteCount", ascending=False)[
    ["Text_Orig", "Text_ML", "voteCount", "replyCount", "Emoji_Sentiment", "source_file"]
].head(20) if "voteCount" in data.columns else pd.DataFrame()

top_replies = data.sort_values("replyCount", ascending=False)[
    ["Text_Orig", "Text_ML", "voteCount", "replyCount", "Emoji_Sentiment", "source_file"]
].head(20) if "replyCount" in data.columns else pd.DataFrame()

engagement_stats, top_likes.head(10), top_replies.head(10)


(           Metric    Value
 0       Avg Likes     3.15
 1    Median Likes     0.00
 2       Max Likes  3300.00
 3     Avg Replies     0.46
 4  Median Replies     0.00
 5     Max Replies   114.00,
                                               Text_Orig  \
 2678                  اطالب ب العم علي يكون العضو رقم16   
 2442  ياخي صالح شخصيته بوجود ابوه عجيييبة كأنه طفل، ...   
 3035  توي في بداية المقطع لكن خانقتني العبرة من طيبة...   
 1081  السلام عليكم ورحمة الله وبركاته  تحيه لبو صالح...   
 2523  كم كنت متعجب في شخصية صالح حتى شفت شخصية ابوه ...   
 7572  جزائري محب للسعودية حبا شديدا ماشاء الله تبارك...   
 2516  27:09\nوالله كسر قلبي العم علي يوم يقول السنه ...   
 1705  بس عرفنا كيف صالح جايب القبول ، من العم علي جع...   
 2376  ما شاء الله تبارك الرحمن\nفواز انسان جداً محتر...   
 7616  كاد قلبي يتوقف\nيا جمالك معشوقتي ״السعودية ״\n...   
 
                                                 Text_ML  voteCount  \
 2678                 اطالب العم علي العضو رقم16 EMO_NEU       3300   

### **Short Comments Stats**

In [33]:
short_stats = pd.DataFrame({
    "Metric": ["Short_TR_%", "Short_ML_%"],
    "Value": [
        round(data["Is_Short_TR"].mean() * 100, 2),
        round(data["Is_Short_ML"].mean() * 100, 2)
    ]
})

short_stats


,Metric,Value
0,Short_TR_%,10.83
1,Short_ML_%,6.91


### **Strong examples Before/After (most changing cases)**


In [34]:
data["Diff_Len"] = (
    data["Text_Orig"].fillna("").astype(str).str.len()
    - data["Text_Base"].fillna("").astype(str).str.len()
).abs()

examples = data.sort_values("Diff_Len", ascending=False).head(15)[
    ["Text_Orig", "Text_Base", "Text_TR", "Text_ML", "Emoji_Sentiment", "Diff_Len", "source_file"]
]

examples


,Text_Orig,Text_Base,Text_TR,Text_ML,Emoji_Sentiment,Diff_Len,source_file
15522,"يقول الإمام العلامة ابن عقيل رحمه الله: ""إذا أ...",يقول الامام العلامة ابن عقيل رحمه الله اذا ارد...,يقول الامام العلامة ابن عقيل رحمه الله اذا ارد...,يقول الامام العلامة ابن عقيل رحمه الله اردت ان...,NEU,1057,Aseer Vedio Comments 12_textready.xlsx
14471,زنجبار كانت عاصمة للامبراطورية العمانية ولولا...,زنجبار كانت عاصمة للامبراطورية العمانية ولولا ...,زنجبار كانت عاصمة للامبراطورية العمانية ولولا ...,زنجبار عاصمة للامبراطورية العمانية ولولا مشارك...,NEU,1017,Najran Vedio Comments 1_textready.xlsx
6444,جيت عادل من جاء عادل . قال ألله تعالى(فَأَمَّا...,جيت عادل من جاء عادل قال الله تعالي فاما الذين...,جيت عادل من جاء عادل قال الله تعالي فاما الذين...,جيت عادل جاء عادل قال الله تعالي فاما امنوا في...,NEU,986,Jazan Vedio Comments 11_textready.xlsx
15222,بِسۡمِ ٱللَّهِ ٱلرَّحۡمَٰنِ ٱلرَّحِيمِ (1) \nٱ...,بسم الله الرحمن الرحيم 1 الحمد لله رب العلمين ...,بسم الله الرحمن الرحيم 1 الحمد لله رب العلمين ...,بسم الله الرحمن الرحيم الحمد لله رب العلمين ال...,NEU,873,Aseer Vedio Comments 10_textready.xlsx
14673,يجب أن نتذكر أننا هنا في هذا العالم لفترة قصير...,يجب ان نتذكر اننا هنا في هذا العالم لفترة قصير...,يجب ان نتذكر اننا هنا في هذا العالم لفترة قصير...,يجب ان نتذكر اننا العالم لفترة قصيرة جدا يجب ا...,NEU,656,Najran Vedio Comments 3_textready.xlsx
18873,وين يوسففففففففففففففففففففففففففففففففففففففف...,وين يوسفغفڤفڤفڤفڤفڤفڤفڤفڤفڤفڤففڤف,وين يوسفغفڤفڤفڤفڤفڤفڤفڤفڤفڤفڤففڤف,وين يوسفغفڤفڤفڤفڤفڤفڤفڤفڤفڤفڤففڤف EMO_NEU,NEU,652,Aseer Vedio Comments 17_textready.xlsx
20593,زورو موقعنا للاتصال او الاستفسار عن شركة تنظيف...,زورو موقعنا للاتصال او الاستفسار عن شركة تنظيف...,زورو موقعنا للاتصال او الاستفسار عن شركة تنظيف...,زورو موقعنا للاتصال او الاستفسار شركة تنظيف خز...,NEU,615,Aseer Vedio Comments 22_textready.xlsx
9299,استغفر الله الذي لا إله الا هو الحي القيوم وأت...,استغفر الله الذي لا اله الا هو الحي القيوم وات...,استغفر الله الذي لا اله الا هو الحي القيوم وات...,استغفر الله لا_اله الا هو الحي القيوم واتوب ال...,POS,600,Jazan Vedio Comments 16_textready.xlsx
9285,‏حافظوا على قول : لا إله إلا أنت سُبحانك إني ك...,حافظوا علي قول لا اله الا انت سبحانك اني كنت م...,حافظوا علي قول لا اله الا انت سبحانك اني كنت م...,حافظوا علي قول لا_اله الا سبحانك اني كنت الظال...,POS,370,Jazan Vedio Comments 16_textready.xlsx
20532,#الدعاء . . \n\nالدعاء عبادة . . !! عليكم بالد...,الدعاء الدعاء عبادة عليكم بالدعاء فيهي تعلوا ا...,الدعاء الدعاء عبادة عليكم بالدعاء فيهي تعلوا ا...,الدعاء الدعاء عبادة عليكم بالدعاء فيهي تعلوا ا...,NEU,342,Aseer Vedio Comments 22_textready.xlsx


### **General statistics + number of files**


In [35]:
overview = pd.DataFrame({
    "Metric": ["Total Comments", "Total Files", "Total Columns"],
    "Value": [len(data), data["source_file"].nunique(), len(data.columns)]
})
overview


,Metric,Value
0,Total Comments,22679
1,Total Files,82
2,Total Columns,22


### **Percentage of empty comments before and after processing**


In [36]:
def empty_ratio(series):
    s = series.fillna("").astype(str).str.strip()
    return round(((s == "") | (s.str.lower().isin(["nan","none"]))).mean() * 100, 2)

empty_stats = pd.DataFrame({
    "Column": ["comment(raw)", "Text_Base", "Text_TR", "Text_ML"],
    "Empty_%": [
        empty_ratio(data.get("comment", pd.Series(dtype=str))),
        empty_ratio(data["Text_Base"]),
        empty_ratio(data["Text_TR"]),
        empty_ratio(data["Text_ML"])
    ]
})
empty_stats


,Column,Empty_%
0,comment(raw),1.67
1,Text_Base,3.71
2,Text_TR,6.32
3,Text_ML,0.00


### **Average length of text before and after (Words + Characters)**


In [37]:
def text_stats(series):
    s = series.fillna("").astype(str).str.strip()
    words = s.apply(lambda x: len(x.split()))
    chars = s.str.len()
    return pd.Series({
        "Avg_Words": round(words.mean(), 2),
        "Median_Words": words.median(),
        "Max_Words": words.max(),
        "Avg_Chars": round(chars.mean(), 2)
    })

length_report = pd.DataFrame({
    "Raw(comment)": text_stats(data.get("comment", pd.Series(dtype=str))),
    "Text_Base": text_stats(data["Text_Base"]),
    "Text_TR": text_stats(data["Text_TR"]),
    "Text_ML": text_stats(data["Text_ML"])
}).T

length_report


,Avg_Words,Median_Words,Max_Words,Avg_Chars
Raw(comment),13.36,8.0,995.0,75.26
Text_Base,13.06,7.0,955.0,69.91
Text_TR,12.66,7.0,954.0,67.69
Text_ML,12.13,7.0,841.0,71.41


### **Emoji Statistics**

In [38]:
emoji_ratio = round((data["Emoji_Count"] > 0).mean() * 100, 2)
emoji_count = int((data["Emoji_Count"] > 0).sum())

emoji_dist = data["Emoji_Sentiment"].value_counts(dropna=False)

emoji_overview = pd.DataFrame({
    "Metric": ["Comments with Emoji %", "Comments with Emoji (count)"],
    "Value": [emoji_ratio, emoji_count]
})

emoji_overview, emoji_dist


(                        Metric    Value
 0        Comments with Emoji %    11.21
 1  Comments with Emoji (count)  2542.00,
 Emoji_Sentiment
 NEU    20617
 POS     2051
 NEG       11
 Name: count, dtype: int64)

### **The 20 most frequently used emojis**


In [39]:
all_emojis = data["Emoji_List"].dropna().sum()
top_emojis = pd.DataFrame(Counter(all_emojis).most_common(20), columns=["Emoji", "Count"])
top_emojis


,Emoji,Count
0,[,22679
1,],22679
2,',11572
3,❤,4837
4,",",3244
5,,3244
6,♥,320
7,✨,99
8,✋,57
9,✌,45


### **Emoji distribution based on interaction (Likes/Replies)**


In [40]:
emoji_engagement = data.groupby("Emoji_Sentiment").agg({
    "voteCount": ["mean", "max"],
    "replyCount": ["mean", "max"],
    "Text_ML": "count"
})

emoji_engagement.columns = ["Avg_Likes", "Max_Likes", "Avg_Replies", "Max_Replies", "Count"]
emoji_engagement.reset_index()


,Emoji_Sentiment,Avg_Likes,Max_Likes,Avg_Replies,Max_Replies,Count
0,NEG,38.909091,395,10.636364,112,11
1,NEU,2.890673,3300,0.446961,114,20617
2,POS,5.525597,1100,0.505607,53,2051


### **Top Words**

In [41]:
words = " ".join(ml_clean).split()
top_words = pd.DataFrame(Counter(words).most_common(30), columns=["Word", "Count"])
top_words


,Word,Count
0,الله,6832
1,علي,3036
2,اليمن,1744
3,ان,1646
4,الي,1447
5,يا,1362
6,والله,1340
7,السعودية,1240
8,بس,1021
9,ولا,998


### **Top Bigrams**

In [42]:
tokens = " ".join(ml_clean).split()
bigrams = list(zip(tokens, tokens[1:]))

top_bigrams = pd.DataFrame(
    [(" ".join(k), v) for k, v in Counter(bigrams).most_common(25)],
    columns=["Bigram", "Count"]
)
top_bigrams


,Bigram,Count
0,ماشاء الله,657
1,تبارك الله,419
2,الله تبارك,407
3,ما_شاء الله,388
4,سبحان الله,299
5,شاء الله,274
6,ان شاء,235
7,الله عليه,224
8,العم علي,213
9,تبارك الرحمن,198


### **Duplicate Comments**

In [45]:
dup_count = base_clean.duplicated().sum()
dup_ratio = round((dup_count / len(data)) * 100, 2)

dup_report = pd.DataFrame({
    "Metric": ["Duplicate Comments", "Duplicate Ratio %"],
    "Value": [int(dup_count), dup_ratio]
})

top_repeated = base_clean.value_counts().head(20).reset_index()
top_repeated.columns = ["Repeated_Comment", "Frequency"]

dup_report, top_repeated


NameError: name 'base_clean' is not defined

### **Engagement Statistics (Likes / Replies)**

In [44]:
engagement_stats = pd.DataFrame({
    "Metric": [
        "Avg Likes", "Median Likes", "Max Likes",
        "Avg Replies", "Median Replies", "Max Replies"
    ],
    "Value": [
        round(data["voteCount"].mean(), 2),
        float(data["voteCount"].median()),
        int(data["voteCount"].max()),
        round(data["replyCount"].mean(), 2),
        float(data["replyCount"].median()),
        int(data["replyCount"].max())
    ]
})
engagement_stats


,Metric,Value
0,Avg Likes,3.15
1,Median Likes,0.00
2,Max Likes,3300.00
3,Avg Replies,0.46
4,Median Replies,0.00
5,Max Replies,114.00


### **Top 20 comments**


In [46]:
top_likes = data.sort_values("voteCount", ascending=False)[
    ["Text_Orig", "Text_ML", "voteCount", "replyCount", "Emoji_Sentiment", "source_file"]
].head(20)

top_likes


,Text_Orig,Text_ML,voteCount,replyCount,Emoji_Sentiment,source_file
2678,اطالب ب العم علي يكون العضو رقم16,اطالب العم علي العضو رقم16 EMO_NEU,3300,65,NEU,Al-Baha Vedio Comments 15_textready.xlsx
2442,ياخي صالح شخصيته بوجود ابوه عجيييبة كأنه طفل، ...,ياخي صالح شخصيته بوجود ابوه عجيبة كانه طفل، وم...,2000,15,NEU,Al-Baha Vedio Comments 15_textready.xlsx
3035,توي في بداية المقطع لكن خانقتني العبرة من طيبة...,توي بداية المقطع لكن خانقتني العبرة طيبة قلب ا...,1100,5,POS,Al-Baha Vedio Comments 15_textready.xlsx
1081,السلام عليكم ورحمة الله وبركاته تحيه لبو صالح...,السلام عليكم ورحمة الله وبركاته تحيه لبو صالح ...,1100,24,NEU,Al-Baha Vedio Comments 15_textready.xlsx
2523,كم كنت متعجب في شخصية صالح حتى شفت شخصية ابوه ...,كم كنت متعجب شخصية صالح حتي شفت شخصية ابوه الل...,932,8,POS,Al-Baha Vedio Comments 15_textready.xlsx
7572,جزائري محب للسعودية حبا شديدا ماشاء الله تبارك...,جزائري محب للسعودية حبا شديدا ماشاء الله تبارك...,737,50,NEU,Jazan Vedio Comments 11_textready.xlsx
2516,27:09\nوالله كسر قلبي العم علي يوم يقول السنه ...,والله كسر قلبي العم علي يوم يقول السنه هذي ما_...,512,2,NEU,Al-Baha Vedio Comments 15_textready.xlsx
1705,بس عرفنا كيف صالح جايب القبول ، من العم علي جع...,بس عرفنا كيف صالح جايب القبول العم علي جعل ربي...,454,3,POS,Al-Baha Vedio Comments 15_textready.xlsx
2376,ما شاء الله تبارك الرحمن\nفواز انسان جداً محتر...,ما_شاء الله تبارك الرحمن فواز انسان جدا محترم ...,451,1,NEU,Al-Baha Vedio Comments 15_textready.xlsx
7616,كاد قلبي يتوقف\nيا جمالك معشوقتي ״السعودية ״\n...,كاد قلبي يتوقف يا جمالك معشوقتي السعودية فلسطي...,449,25,NEU,Jazan Vedio Comments 11_textready.xlsx


### **Top 20 Viral Comments (Replies)**


In [47]:
top_replies = data.sort_values("replyCount", ascending=False)[
    ["Text_Orig", "Text_ML", "voteCount", "replyCount", "Emoji_Sentiment", "source_file"]
].head(20)

top_replies


,Text_Orig,Text_ML,voteCount,replyCount,Emoji_Sentiment,source_file
12207,الله على الاراضي السعودية وجمالها ، دمتي عزا ...,الله علي الاراضي السعودية وجمالها دمتي عزا وفخ...,141,114,NEU,Jazan Vedio Comments 3_textready.xlsx
6160,ليت اهل المنطقة يستثمرونها بالسياحة مناظر جميل...,ليت اهل المنطقة يستثمرونها بالسياحة مناظر جميل...,395,112,NEG,Jazan Vedio Comments 11_textready.xlsx
15854,والله السعوديين عرب حق وحقيق واعتزازهم بثقافته...,والله السعوديين عرب حق وحقيق واعتزازهم بثقافته...,312,104,NEU,Aseer Vedio Comments 12_textready.xlsx
17305,رائعة ماكنت اتخيل هالجمال بالسعودية كنت أظنها ...,رائعة ماكنت اتخيل هالجمال بالسعودية كنت اظنها ...,214,92,NEU,Aseer Vedio Comments 14_textready.xlsx
16193,جنوب السعوديه هو اليمن بلد عندها حضارة سبأ وهم...,جنوب السعوديه هو اليمن بلد عندها حضارة سبا وهم...,10,82,NEU,Aseer Vedio Comments 12_textready.xlsx
16059,السعودية بلد الحرمين والتاريخ والحضارة والتنوع...,السعودية بلد الحرمين والتاريخ والحضارة والتنوع...,444,81,NEU,Aseer Vedio Comments 12_textready.xlsx
7079,ماشاء الله هل هذه حقيقة أم خيال؟ نحن في الجزائ...,ماشاء الله هل حقيقة ام خيال؟ الجزائر نعرف السع...,291,79,NEU,Jazan Vedio Comments 11_textready.xlsx
8103,جنوب السعوديه ؟! \nليش ماتكتب جيزان .. والا مس...,جنوب السعوديه ليش ماتكتب جيزان والا مسوي نفسك ...,3,75,NEU,Jazan Vedio Comments 11_textready.xlsx
12579,بسبب قربنا من اليمن البعض منهم ينسبونا لهم الج...,بسبب قربنا اليمن البعض منهم ينسبونا لهم الجنوب...,64,68,NEU,Jazan Vedio Comments 3_textready.xlsx
2678,اطالب ب العم علي يكون العضو رقم16,اطالب العم علي العضو رقم16 EMO_NEU,3300,65,NEU,Al-Baha Vedio Comments 15_textready.xlsx


### **Short Comments Statistics**

In [48]:
short_stats = pd.DataFrame({
    "Metric": ["Short_TR_%", "Short_ML_%"],
    "Value": [
        round(data["Is_Short_TR"].mean() * 100, 2),
        round(data["Is_Short_ML"].mean() * 100, 2)
    ]
})
short_stats


,Metric,Value
0,Short_TR_%,10.83
1,Short_ML_%,6.91


### **(Negation Analysis)**

In [49]:
neg_words = ["ما", "لا", "لم", "لن", "ليس", "مو", "مش", "بدون"]

neg_counts = {w: data["Text_TR"].fillna("").astype(str).str.contains(rf"\b{w}\b", regex=True).sum()
              for w in neg_words}

neg_df = pd.DataFrame(list(neg_counts.items()), columns=["Negation_Word", "Count"]) \
           .sort_values("Count", ascending=False)

neg_df


,Negation_Word,Count
0,ما,1522
1,لا,1146
5,مو,341
2,لم,313
7,بدون,144
6,مش,101
4,ليس,100
3,لن,42


### **Hashtags / Mentions / URLs within raw comments**


#### **Hashtags**

In [50]:
hashtags = data["comment"].fillna("").astype(str).str.findall(r"#(\w+)")
hashtags_flat = hashtags.explode().dropna()

top_hashtags = hashtags_flat.value_counts().head(20).reset_index()
top_hashtags.columns = ["Hashtag", "Count"]

top_hashtags


,Hashtag,Count
0,ساعة_استجابة,2
1,جازان,2
2,القناة,2
3,العراق,2
4,ملاحظة,1
5,ديسلايك,1
6,الدعاء,1
7,العبادة_العظيمة,1
8,الله_يريد,1
9,شروط_الدعاء,1


#### **Mentions**

In [51]:
mentions = data["comment"].fillna("").astype(str).str.findall(r"@(\w+)")
mentions_flat = mentions.explode().dropna()

top_mentions = mentions_flat.value_counts().head(20).reset_index()
top_mentions.columns = ["Mention", "Count"]

top_mentions


,Mention,Count
0,aghrman,124
1,عزالدين129,111
2,user,68
3,OokKkk,52
4,احمدصالح,48
5,YasminMahmoud,30
6,Khal1222,27
7,احمدالجابىي,23
8,الشجاعهابينيه,22
9,نوفلالحظرمي,20


#### **URLs**

In [52]:
url_count = data["comment"].fillna("").astype(str).str.contains(r"https?://|www\.", regex=True).sum()
print("Comments containing URLs:", url_count)
print("URL ratio %:", round((url_count/len(data))*100, 2))


Comments containing URLs: 121
URL ratio %: 0.53


### **Before/After examples**


In [53]:
data["Diff_Len"] = (
    data["Text_Orig"].fillna("").astype(str).str.len()
    - data["Text_Base"].fillna("").astype(str).str.len()
).abs()

examples = data.sort_values("Diff_Len", ascending=False).head(15)[
    ["Text_Orig", "Text_Base", "Text_TR", "Text_ML", "Emoji_Sentiment", "Diff_Len", "source_file"]
]

examples


,Text_Orig,Text_Base,Text_TR,Text_ML,Emoji_Sentiment,Diff_Len,source_file
15522,"يقول الإمام العلامة ابن عقيل رحمه الله: ""إذا أ...",يقول الامام العلامة ابن عقيل رحمه الله اذا ارد...,يقول الامام العلامة ابن عقيل رحمه الله اذا ارد...,يقول الامام العلامة ابن عقيل رحمه الله اردت ان...,NEU,1057,Aseer Vedio Comments 12_textready.xlsx
14471,زنجبار كانت عاصمة للامبراطورية العمانية ولولا...,زنجبار كانت عاصمة للامبراطورية العمانية ولولا ...,زنجبار كانت عاصمة للامبراطورية العمانية ولولا ...,زنجبار عاصمة للامبراطورية العمانية ولولا مشارك...,NEU,1017,Najran Vedio Comments 1_textready.xlsx
6444,جيت عادل من جاء عادل . قال ألله تعالى(فَأَمَّا...,جيت عادل من جاء عادل قال الله تعالي فاما الذين...,جيت عادل من جاء عادل قال الله تعالي فاما الذين...,جيت عادل جاء عادل قال الله تعالي فاما امنوا في...,NEU,986,Jazan Vedio Comments 11_textready.xlsx
15222,بِسۡمِ ٱللَّهِ ٱلرَّحۡمَٰنِ ٱلرَّحِيمِ (1) \nٱ...,بسم الله الرحمن الرحيم 1 الحمد لله رب العلمين ...,بسم الله الرحمن الرحيم 1 الحمد لله رب العلمين ...,بسم الله الرحمن الرحيم الحمد لله رب العلمين ال...,NEU,873,Aseer Vedio Comments 10_textready.xlsx
14673,يجب أن نتذكر أننا هنا في هذا العالم لفترة قصير...,يجب ان نتذكر اننا هنا في هذا العالم لفترة قصير...,يجب ان نتذكر اننا هنا في هذا العالم لفترة قصير...,يجب ان نتذكر اننا العالم لفترة قصيرة جدا يجب ا...,NEU,656,Najran Vedio Comments 3_textready.xlsx
18873,وين يوسففففففففففففففففففففففففففففففففففففففف...,وين يوسفغفڤفڤفڤفڤفڤفڤفڤفڤفڤفڤففڤف,وين يوسفغفڤفڤفڤفڤفڤفڤفڤفڤفڤفڤففڤف,وين يوسفغفڤفڤفڤفڤفڤفڤفڤفڤفڤفڤففڤف EMO_NEU,NEU,652,Aseer Vedio Comments 17_textready.xlsx
20593,زورو موقعنا للاتصال او الاستفسار عن شركة تنظيف...,زورو موقعنا للاتصال او الاستفسار عن شركة تنظيف...,زورو موقعنا للاتصال او الاستفسار عن شركة تنظيف...,زورو موقعنا للاتصال او الاستفسار شركة تنظيف خز...,NEU,615,Aseer Vedio Comments 22_textready.xlsx
9299,استغفر الله الذي لا إله الا هو الحي القيوم وأت...,استغفر الله الذي لا اله الا هو الحي القيوم وات...,استغفر الله الذي لا اله الا هو الحي القيوم وات...,استغفر الله لا_اله الا هو الحي القيوم واتوب ال...,POS,600,Jazan Vedio Comments 16_textready.xlsx
9285,‏حافظوا على قول : لا إله إلا أنت سُبحانك إني ك...,حافظوا علي قول لا اله الا انت سبحانك اني كنت م...,حافظوا علي قول لا اله الا انت سبحانك اني كنت م...,حافظوا علي قول لا_اله الا سبحانك اني كنت الظال...,POS,370,Jazan Vedio Comments 16_textready.xlsx
20532,#الدعاء . . \n\nالدعاء عبادة . . !! عليكم بالد...,الدعاء الدعاء عبادة عليكم بالدعاء فيهي تعلوا ا...,الدعاء الدعاء عبادة عليكم بالدعاء فيهي تعلوا ا...,الدعاء الدعاء عبادة عليكم بالدعاء فيهي تعلوا ا...,NEU,342,Aseer Vedio Comments 22_textready.xlsx
